# **Stage 1 - Exploratory Data Analysis — Multi-Dataset**
## US Quarterly · US Monthly · UK Quarterly
___
**All variables are retained throughout. No rows or columns are ever dropped.**
`FEATURES_*` lists control what enters each analytical step.
Exclusion rationale is documented in Section 2 and summarised in Section 8.

| Dataset | File | Freq | Target |
|---------|------|------|--------|
| US Quarterly | `MacroVariables_US_Q.csv` | Q | `us_delinquency_rate` |
| US Monthly | `MacroVariables_US_M.csv` | M | `us_delinquency_rate` (spline interpolated from quarterly) |
| UK Quarterly | `MacroVariables_UK_Q.csv` | Q | None — Stage 1 macro forecasting only |

**References:** Bank & Eder (2021) SSRN 3981339 · Djeundje & Crook (2019) EJOR 275(1) · Breeden & Crook (2020) JORS


## **0: Imports & Configuration**
___

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from statsmodels.tsa.stattools import adfuller, kpss, ccf, acf, pacf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram as scipy_dendrogram
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.precision', 3)

# ── Colour palette ────────────────────────────────────────────────────────────
TEMPLATE = 'plotly_white'
NAVY, BLUE, RED, GREEN, AMBER, GREY, LBLUE = (
    '#1F3864', '#2E75B6', '#C0392B', '#27AE60', '#E67E22', '#7F8C8D', '#AED6F1')

# ── Recession bands ───────────────────────────────────────────────────────────
RECESSIONS_US = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]
RECESSIONS_UK = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2008-01-01', '2009-06-30', 'GFC'),
    ('2020-01-01', '2020-06-30', 'COVID'),
]

def add_recessions(fig, recessions, row=None, col=None):
    kw = dict(row=row, col=col) if row is not None else {}
    for s, e, lbl in recessions:
        fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.35,
                      layer='below', line_width=0, annotation_text=lbl,
                      annotation_position='top left', annotation_font_size=8,
                      annotation_font_color=GREY, **kw)
    return fig

# ── Table styler functions ────────────────────────────────────────────────────
def c_missing(v):
    if v == 0:   return 'background-color:#D5F5E3; color:#1a1a1a'
    if v <= 5:   return 'background-color:#FEF9E7; color:#1a1a1a'
    if v <= 15:  return 'background-color:#FDEBD0; color:#1a1a1a'
    return              'background-color:#FADBD8; color:#1a1a1a'
def c_decision(v):
    m = {'Use as-is': '#D5F5E3', 'Transform': '#FEF9E7', 'Exclude': '#FADBD8', 'Pending': '#EBF5FB'}
    return f'background-color:{m.get(v, "#FDEBD0")}; color:#1a1a1a'
def c_stat(v):
    m = {'STATIONARY': '#D5F5E3', 'NON-STATIONARY': '#FADBD8',
         'INCONCLUSIVE': '#FEF9E7', 'CONFLICTING': '#FDEBD0'}
    return f'background-color:{m.get(v, "white")}; color:#1a1a1a'
def c_p(v):
    if v < 0.01: return 'background-color:#D5F5E3; color:#1a1a1a'
    if v < 0.05: return 'background-color:#FEF9E7; color:#1a1a1a'
    if v < 0.10: return 'background-color:#FDEBD0; color:#1a1a1a'
    return              'background-color:#FADBD8; color:#1a1a1a'
def c_skew(v):
    if abs(v) > 2: return 'background-color:#FADBD8; color:#1a1a1a'
    if abs(v) > 1: return 'background-color:#FEF9E7; color:#1a1a1a'
    return                'background-color:#D5F5E3; color:#1a1a1a'
def c_kurt(v):
    if abs(v) > 5: return 'background-color:#FADBD8; color:#1a1a1a'
    if abs(v) > 3: return 'background-color:#FEF9E7; color:#1a1a1a'
    return                'background-color:#D5F5E3; color:#1a1a1a'
def c_out(v):
    if v > 5: return 'background-color:#FADBD8; color:#1a1a1a'
    if v > 2: return 'background-color:#FEF9E7; color:#1a1a1a'
    return           'background-color:#D5F5E3; color:#1a1a1a'
def c_sig(v):
    if v == 'Yes': return 'background-color:#D5F5E3; color:#1a1a1a; font-weight:bold'
    return                'background-color:#FADBD8; color:#1a1a1a'
def c_dir(v):
    if v == 'Positive': return 'background-color:#FDEBD0; color:#1a1a1a'
    return                     'background-color:#EBF5FB; color:#1a1a1a'
def c_vif(v):
    if v < 5:  return 'background-color:#D5F5E3; color:#1a1a1a'
    if v < 10: return 'background-color:#FEF9E7; color:#1a1a1a'
    return            'background-color:#FADBD8; color:#1a1a1a; font-weight:bold'
def c_grp(v):
    if 'Non' in str(v): return 'background-color:#FADBD8; color:#7F8C8D'
    return                     'background-color:#EBF5FB; color:#1a1a1a'
def c_confirmed(v):
    if v == 'Yes':   return 'background-color:#D5F5E3; color:#1a1a1a'
    if v == 'Check': return 'background-color:#FADBD8; color:#1a1a1a'
    return ''
def c_action(v):
    if 'stress' in str(v): return 'background-color:#FEF9E7; color:#1a1a1a'
    if v == 'Investigate':  return 'background-color:#FADBD8; color:#1a1a1a'
    return                         'background-color:#D5F5E3; color:#1a1a1a'
def c_pattern(v):
    if 'AR process'   in str(v): return 'background-color:#EBF5FB; color:#1a1a1a'
    if 'MA process'   in str(v): return 'background-color:#FEF9E7; color:#1a1a1a'
    if 'ARMA process' in str(v): return 'background-color:#FDEBD0; color:#1a1a1a'
    return                              'background-color:#D5F5E3; color:#1a1a1a'
def c_siglags(v):
    if v >= 10: return 'background-color:#FADBD8; color:#1a1a1a'
    if v >= 5:  return 'background-color:#FEF9E7; color:#1a1a1a'
    return             'background-color:#D5F5E3; color:#1a1a1a'

# ── File paths ────────────────────────────────────────────────────────────────
# Update BASE to match your local directory if needed
BASE = Path(r"C:\Users\andre\Downloads")
PATHS = {
    'US_Q': BASE / 'MacroVariables_US_Q.csv',
    'US_M': BASE / 'MacroVariables_US_M.csv',
    'UK_Q': BASE / 'MacroVariables_UK_Q.csv',
}

# ── Sample window ─────────────────────────────────────────────────────────────
WINDOW_START = '1990-01-01'
WINDOW_END   = None  # update to a specific date string if you want to cap the window

print('Configuration loaded.')
for k, p in PATHS.items():
    print(f'  {k}: {p}')

Configuration loaded.
  US_Q: C:\Users\andre\Downloads\MacroVariables_US_Q.csv
  US_M: C:\Users\andre\Downloads\MacroVariables_US_M.csv
  UK_Q: C:\Users\andre\Downloads\MacroVariables_UK_Q.csv


## **1: Data Loading — Raw Dataset Audit**
___
This section documents the datasets **as loaded from the CSVs** — before any derivations.
Each completeness table and overview plot shows only the native columns that came from the
Data Collection pipeline. Derived variables are added in Section 2.

The Data Collection pipeline already drops the current incomplete quarter/month, so no
end-date filtering is needed — `WINDOW_END = None` uses the full available history.


### 1a — US Quarterly

In [2]:
TARGET_USQ    = 'us_delinquency_rate'
PREFIX_USQ    = 'us_'
RECESSIONS_USQ = RECESSIONS_US

raw_usq = pd.read_csv(PATHS['US_Q'], index_col=0, parse_dates=True)
raw_usq.index = pd.DatetimeIndex(raw_usq.index).to_period('Q').to_timestamp('Q')
raw_usq = raw_usq.sort_index()
df_usq  = raw_usq.loc[WINDOW_START:].copy()

print(f'Shape: {df_usq.shape}  |  {df_usq.index.min().date()} to {df_usq.index.max().date()}')
print(f'Columns ({df_usq.shape[1]}): {df_usq.columns.tolist()}')

t = df_usq[TARGET_USQ].dropna()
print(f'\nTarget — {TARGET_USQ}: {len(t)} obs | Min {t.min():.2f}% | Max {t.max():.2f}% | Mean {t.mean():.2f}%')

completeness = pd.DataFrame({
    'Non-null':    df_usq.notna().sum(),
    'Missing':     df_usq.isna().sum(),
    'Missing (%)': (df_usq.isna().sum() / len(df_usq) * 100).round(1),
    'First valid': df_usq.apply(lambda s: (
        f'{s.first_valid_index().year}-Q{s.first_valid_index().quarter}'
        if s.first_valid_index() is not None else None)),
    'Last valid':  df_usq.apply(lambda s: (
        f'{s.last_valid_index().year}-Q{s.last_valid_index().quarter}'
        if s.last_valid_index() is not None else None)),
})
display(completeness.style
        .map(c_missing, subset=['Missing (%)'])
        .format({'Missing (%)': '{:.1f}'})
        .set_caption('Table 1a.1 — Completeness: US Quarterly (raw dataset)'))


Shape: (144, 22)  |  1990-03-31 to 2025-12-31
Columns (22): ['us_house_price_idx', 'us_cpi', 'us_unemployment', 'us_consumer_confidence', 'us_real_gdp', 'us_gdp_yoy_growth', 'us_bond_yield_10y', 'us_bond_yield_1y', 'us_term_spread', 'us_reer', 'us_reer_log_ret', 'us_oil_price', 'us_oil_log_ret', 'us_credit', 'us_credit_qoq_growth', 'us_credit_yoy_growth', 'us_industrial_production', 'us_vix', 'us_vix_log_ret', 'us_delinquency_rate', 'us_sp500_close', 'us_sp500_log_ret']

Target — us_delinquency_rate: 140 obs | Min 1.53% | Max 6.77% | Mean 3.70%


,Non-null,Missing,Missing (%),First valid,Last valid
us_house_price_idx,144,0,0.0,1990-Q1,2025-Q4
us_cpi,143,1,0.7,1990-Q1,2025-Q3
us_unemployment,144,0,0.0,1990-Q1,2025-Q4
us_consumer_confidence,144,0,0.0,1990-Q1,2025-Q4
us_real_gdp,144,0,0.0,1990-Q1,2025-Q4
us_gdp_yoy_growth,144,0,0.0,1990-Q1,2025-Q4
us_bond_yield_10y,144,0,0.0,1990-Q1,2025-Q4
us_bond_yield_1y,144,0,0.0,1990-Q1,2025-Q4
us_term_spread,144,0,0.0,1990-Q1,2025-Q4
us_reer,128,16,11.1,1994-Q1,2025-Q4


In [3]:
# Figure 1a.1 — Target variable time series
mean_val = df_usq[TARGET_USQ].mean()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_usq.index, y=df_usq[TARGET_USQ], mode='lines',
    line=dict(color=NAVY, width=2.5), fill='tozeroy',
    fillcolor='rgba(46,117,182,0.10)',
    hovertemplate='%{x|%Y Q%q}<br>%{y:.2f}%<extra></extra>'))
fig.add_hline(y=mean_val, line_dash='dash', line_color=GREY, line_width=1.5,
              annotation_text=f'Mean={mean_val:.2f}%',
              annotation_position='top right', annotation_font_color=GREY)
fig = add_recessions(fig, RECESSIONS_USQ)
for s, e in [('2007', '2010'), ('2019', '2021')]:
    sub = df_usq[TARGET_USQ].loc[s:e].dropna()
    if not sub.empty:
        fig.add_annotation(x=sub.idxmax(), y=sub.max(), text=f'{sub.max():.1f}%',
            showarrow=True, arrowhead=2, arrowcolor=RED,
            font=dict(color=RED, size=11), bgcolor='white',
            bordercolor=RED, borderwidth=1, ay=-45)
fig.update_layout(
    title=dict(text='Figure 1a.1 — Delinquency Rate: US Quarterly', font=dict(size=14, color=NAVY)),
    xaxis=dict(showgrid=True, gridcolor='#E0E0E0'),
    yaxis=dict(title='Delinquency Rate (%)', showgrid=True, gridcolor='#E0E0E0'),
    template=TEMPLATE, height=430, margin=dict(t=80, b=40, l=60, r=40))
fig.show()

# Figure 1a.2 — All raw variables overview (native columns only, before Section 2 derivations)
raw_cols = [c for c in raw_usq.columns if c != TARGET_USQ]
ncols = 4
nrows = int(np.ceil(len(raw_cols) / ncols))
fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace('us_', '').replace('_', ' ').title() for v in raw_cols],
    vertical_spacing=0.06, horizontal_spacing=0.04)
for i, var in enumerate(raw_cols):
    r, c = i // ncols + 1, i % ncols + 1
    fig2.add_trace(go.Scatter(
        x=df_usq.index, y=df_usq[var], mode='lines',
        line=dict(color=BLUE, width=1.0), showlegend=False), row=r, col=c)
fig2.update_xaxes(showticklabels=False)
fig2.update_yaxes(showticklabels=False)
fig2.update_layout(
    title=dict(text=f'Figure 1a.2 — All Raw Variables: US Quarterly ({len(raw_cols)} series)',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=nrows * 180,
    margin=dict(t=50, b=20, l=20, r=20))
fig2.show()


### 1b — US Monthly

In [4]:
TARGET_USM    = 'us_delinquency_rate'
PREFIX_USM    = 'us_'
RECESSIONS_USM = RECESSIONS_US

raw_usm = pd.read_csv(PATHS['US_M'], index_col=0, parse_dates=True)
raw_usm.index = pd.DatetimeIndex(raw_usm.index).to_period('M').to_timestamp('M')
raw_usm = raw_usm.sort_index()
df_usm  = raw_usm.loc[WINDOW_START:].copy()

print(f'Shape: {df_usm.shape}  |  {df_usm.index.min().date()} to {df_usm.index.max().date()}')
print(f'Columns ({df_usm.shape[1]}): {df_usm.columns.tolist()}')

t = df_usm[TARGET_USM].dropna()
print(f'\nTarget — {TARGET_USM}: {len(t)} obs | Min {t.min():.2f}% | Max {t.max():.2f}% | Mean {t.mean():.2f}%')

completeness = pd.DataFrame({
    'Non-null':    df_usm.notna().sum(),
    'Missing':     df_usm.isna().sum(),
    'Missing (%)': (df_usm.isna().sum() / len(df_usm) * 100),
    'First valid': df_usm.apply(lambda s: (
        s.first_valid_index().strftime('%Y-%m') if s.first_valid_index() is not None else None)),
    'Last valid':  df_usm.apply(lambda s: (
        s.last_valid_index().strftime('%Y-%m')  if s.last_valid_index()  is not None else None)),
})
display(completeness.style
        .map(c_missing, subset=['Missing (%)'])
        .format({'Missing (%)': '{:.1f}'})
        .set_caption('Table 1b.1 — Completeness: US Monthly (raw dataset)'))

mean_val = df_usm[TARGET_USM].mean()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_usm.index, y=df_usm[TARGET_USM], mode='lines',
    line=dict(color=NAVY, width=1.8), fill='tozeroy',
    fillcolor='rgba(46,117,182,0.10)',
    hovertemplate='%{x|%Y-%m}<br>%{y:.2f}%<extra></extra>'))
fig.add_hline(y=mean_val, line_dash='dash', line_color=GREY, line_width=1.5,
              annotation_text=f'Mean={mean_val:.2f}%',
              annotation_position='top right', annotation_font_color=GREY)
fig = add_recessions(fig, RECESSIONS_USM)
fig.update_layout(
    title=dict(
        text='Figure 1b.1 — Delinquency Rate: US Monthly (spline interpolated from quarterly)',
        font=dict(size=13, color=NAVY)),
    xaxis=dict(showgrid=True, gridcolor='#E0E0E0'),
    yaxis=dict(title='Delinquency Rate (%)', showgrid=True, gridcolor='#E0E0E0'),
    template=TEMPLATE, height=430, margin=dict(t=80, b=40, l=60, r=40))
fig.show()

raw_cols = [c for c in raw_usm.columns if c != TARGET_USM]
ncols = 4
nrows = int(np.ceil(len(raw_cols) / ncols))
fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace('us_', '').replace('_', ' ').title() for v in raw_cols],
    vertical_spacing=0.06, horizontal_spacing=0.04)
for i, var in enumerate(raw_cols):
    r, c = i // ncols + 1, i % ncols + 1
    fig2.add_trace(go.Scatter(
        x=df_usm.index, y=df_usm[var], mode='lines',
        line=dict(color=BLUE, width=1.0), showlegend=False), row=r, col=c)
fig2.update_xaxes(showticklabels=False)
fig2.update_yaxes(showticklabels=False)
fig2.update_layout(
    title=dict(text=f'Figure 1b.2 — All Raw Variables: US Monthly ({len(raw_cols)} series)',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=nrows * 180,
    margin=dict(t=50, b=20, l=20, r=20))
fig2.show()


Shape: (434, 22)  |  1990-01-31 to 2026-02-28
Columns (22): ['us_real_gdp', 'us_gdp_yoy_growth', 'us_bond_yield_10y', 'us_bond_yield_1y', 'us_term_spread', 'us_reer', 'us_reer_log_ret', 'us_oil_price', 'us_oil_log_ret', 'us_credit', 'us_credit_mom_growth', 'us_credit_yoy_growth', 'us_industrial_production', 'us_vix', 'us_vix_log_ret', 'us_sp500_close', 'us_sp500_log_ret', 'us_house_price_idx', 'us_cpi', 'us_unemployment', 'us_consumer_confidence', 'us_delinquency_rate']

Target — us_delinquency_rate: 418 obs | Min 1.53% | Max 6.77% | Mean 3.70%


,Non-null,Missing,Missing (%),First valid,Last valid
us_real_gdp,432,2,0.5,1990-01,2025-12
us_gdp_yoy_growth,432,2,0.5,1990-01,2025-12
us_bond_yield_10y,434,0,0.0,1990-01,2026-02
us_bond_yield_1y,434,0,0.0,1990-01,2026-02
us_term_spread,434,0,0.0,1990-01,2026-02
us_reer,386,48,11.1,1994-01,2026-02
us_reer_log_ret,385,49,11.3,1994-02,2026-02
us_oil_price,434,0,0.0,1990-01,2026-02
us_oil_log_ret,434,0,0.0,1990-01,2026-02
us_credit,429,5,1.2,1990-01,2025-09


### 1c — UK Quarterly

**Note:** No UK aggregate default series is available at quarterly frequency back to 1990.
This dataset supports Stage 1 macro forecasting only. Sections requiring a target variable
use `uk_gdp_yoy_growth` as a reference.


In [5]:
TARGET_UKQ    = None
PREFIX_UKQ    = 'uk_'
RECESSIONS_UKQ = RECESSIONS_UK

raw_ukq = pd.read_csv(PATHS['UK_Q'], index_col=0, parse_dates=True)
raw_ukq.index = pd.DatetimeIndex(raw_ukq.index).to_period('Q').to_timestamp('Q')
raw_ukq = raw_ukq.sort_index()
df_ukq  = raw_ukq.loc[WINDOW_START:].copy()

print(f'Shape: {df_ukq.shape}  |  {df_ukq.index.min().date()} to {df_ukq.index.max().date()}')
print(f'Columns ({df_ukq.shape[1]}): {df_ukq.columns.tolist()}')

completeness = pd.DataFrame({
    'Non-null':    df_ukq.notna().sum(),
    'Missing':     df_ukq.isna().sum(),
    'Missing (%)': (df_ukq.isna().sum() / len(df_ukq) * 100).round(1),
    'First valid': df_ukq.apply(lambda s: (
        f'{s.first_valid_index().year}-Q{s.first_valid_index().quarter}'
        if s.first_valid_index() is not None else None)),
    'Last valid':  df_ukq.apply(lambda s: (
        f'{s.last_valid_index().year}-Q{s.last_valid_index().quarter}'
        if s.last_valid_index() is not None else None)),
})
display(completeness.style
        .map(c_missing, subset=['Missing (%)'])
        .format({'Missing (%)': '{:.1f}'})
        .set_caption('Table 1c.1 — Completeness: UK Quarterly (raw dataset)'))

raw_cols = list(raw_ukq.columns)
ncols = 4
nrows = int(np.ceil(len(raw_cols) / ncols))
fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace('uk_', '').replace('_', ' ').title() for v in raw_cols],
    vertical_spacing=0.06, horizontal_spacing=0.04)
for i, var in enumerate(raw_cols):
    r, c = i // ncols + 1, i % ncols + 1
    fig2.add_trace(go.Scatter(
        x=df_ukq.index, y=df_ukq[var], mode='lines',
        line=dict(color=BLUE, width=1.0), showlegend=False), row=r, col=c)
fig2.update_xaxes(showticklabels=False)
fig2.update_yaxes(showticklabels=False)
fig2.update_layout(
    title=dict(text=f'Figure 1c.1 — All Raw Variables: UK Quarterly ({len(raw_cols)} series)',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=nrows * 180,
    margin=dict(t=50, b=20, l=20, r=20))
fig2.show()

Shape: (144, 15)  |  1990-03-31 to 2025-12-31
Columns (15): ['uk_house_price_idx', 'uk_cpi', 'uk_unemployment', 'uk_consumer_confidence', 'uk_real_gdp', 'uk_gdp_yoy_growth', 'uk_bond_yield_10y', 'uk_reer', 'uk_oil_price', 'uk_credit', 'uk_credit_qoq_growth', 'uk_credit_yoy_growth', 'uk_industrial_production', 'uk_ftse_close', 'uk_ftse_qoq_growth']


,Non-null,Missing,Missing (%),First valid,Last valid
uk_house_price_idx,144,0,0.0,1990-Q1,2025-Q4
uk_cpi,144,0,0.0,1990-Q1,2025-Q4
uk_unemployment,144,0,0.0,1990-Q1,2025-Q4
uk_consumer_confidence,144,0,0.0,1990-Q1,2025-Q4
uk_real_gdp,144,0,0.0,1990-Q1,2025-Q4
uk_gdp_yoy_growth,144,0,0.0,1990-Q1,2025-Q4
uk_bond_yield_10y,144,0,0.0,1990-Q1,2025-Q4
uk_reer,128,16,11.1,1994-Q1,2025-Q4
uk_oil_price,144,0,0.0,1990-Q1,2025-Q4
uk_credit,143,1,0.7,1990-Q1,2025-Q3


## **2: Variable Classification & Feature Construction**
___
Derived variables are added here as new columns. Nothing is removed from the dataframes. `FEATURES_*` lists define what enters the analytical pipeline.

**Why derive new variables?**
Raw level variables like `us_house_price_idx` or `us_industrial_production` are non-stationary in levels — they trend upward over time without a stable mean, which violates regression assumptions. Taking YoY growth rates or first differences produces stationary series that carry the same economic signal in a regression-compatible form.

**Derived variables introduced in this section:**
- `{prefix}house_price_yoy` — YoY % change of house price index
- `{prefix}indprod_yoy` — YoY % change of industrial production
- `{prefix}oil_yoy` — YoY % change of oil price
- `{prefix}reer_interp` — REER linearly interpolated over the 1990–1993 gap (intermediate)
- `{prefix}reer_diff` — First difference of interpolated REER

The **variable register table** documents every column in the dataframe — which enter the pipeline and why others are excluded. All excluded variables are retained in the dataframe throughout.

### 2a — US Quarterly

In [6]:
# Derived columns — added to df_usq, originals kept
df_usq['us_house_price_yoy'] = df_usq['us_house_price_idx'].pct_change(4) * 100
df_usq['us_indprod_yoy']     = df_usq['us_industrial_production'].pct_change(4) * 100
df_usq['us_oil_yoy']         = df_usq['us_oil_price'].pct_change(4) * 100
df_usq['us_reer_interp']     = df_usq['us_reer'].interpolate(method='linear', limit_direction='both')
df_usq['us_reer_diff']       = df_usq['us_reer_interp'].diff()

print('Derived columns added to df_usq:')
for c in ['us_house_price_yoy', 'us_indprod_yoy', 'us_oil_yoy', 'us_reer_interp', 'us_reer_diff']:
    print(f'  {c:30s}  non-null: {df_usq[c].notna().sum()}')
print(f'Total columns now: {df_usq.shape[1]}  (was {raw_usq.shape[1]})')

Derived columns added to df_usq:
  us_house_price_yoy              non-null: 140
  us_indprod_yoy                  non-null: 140
  us_oil_yoy                      non-null: 140
  us_reer_interp                  non-null: 144
  us_reer_diff                    non-null: 143
Total columns now: 27  (was 22)


In [7]:
FEATURES_USQ = [
    'us_gdp_yoy_growth',      # YoY GDP — stationary companion to us_real_gdp
    'us_unemployment',         # Cyclical rate
    'us_cpi',                  # YoY inflation
    'us_consumer_confidence',  # OECD amplitude-adjusted index
    'us_bond_yield_10y',       # 10-year Treasury — tested with ct spec in Section 3
    'us_credit_qoq_growth',    # QoQ credit growth — companion to us_credit
    'us_sp500_log_ret',        # S&P 500 log return — native in dataset
    'us_vix_log_ret',          # VIX log return — native in dataset
    'us_house_price_yoy',      # YoY house price — derived above
    'us_indprod_yoy',          # YoY industrial production — derived above
    'us_oil_yoy',              # YoY WTI oil price — derived above
    'us_reer_diff',            # First diff of REER — derived above
]

EXCL_USQ = {
    'us_real_gdp':              'Raw data — replaced by us_gdp_yoy_growth',
    'us_credit':                'Raw data — replaced by us_credit_qoq_growth',
    'us_sp500_close':           'Raw data — replaced by us_sp500_log_ret',
    'us_house_price_idx':       'Raw data — replaced by us_house_price_yoy',
    'us_industrial_production': 'Raw data — replaced by us_indprod_yoy',
    'us_oil_price':             'Raw data — replaced by us_oil_yoy',
    'us_vix':                   'Raw data — replaced by us_vix_log_ret',
    'us_reer':                  'Raw data — replaced by us_reer_diff',
    'us_credit_yoy_growth':     'Transformation — us_credit_qoq_growth preferred at quarterly freq',
    'us_reer_log_ret':          'Transformation — us_reer_diff preferred for consistency',
    'us_oil_log_ret':           'Transformation — us_oil_yoy preferred for interpretability',
    'us_bond_yield_1y':         'Redundant — collinear with us_bond_yield_10y',
    'us_term_spread':           'Redundant — fully determined by bond_yield_10y and bond_yield_1y',
    'us_reer_interp':           'Intermediate — derivation step for us_reer_diff only',
}

df_model_usq = df_usq.copy()

print(f'FEATURES_USQ ({len(FEATURES_USQ)} in pipeline):')
for f in FEATURES_USQ:
    print(f'  {"OK" if f in df_model_usq.columns else "MISSING"}  {f}')
print(f'\ndf_model_usq: {df_model_usq.shape} — all columns retained')

FEATURES_USQ (12 in pipeline):
  OK  us_gdp_yoy_growth
  OK  us_unemployment
  OK  us_cpi
  OK  us_consumer_confidence
  OK  us_bond_yield_10y
  OK  us_credit_qoq_growth
  OK  us_sp500_log_ret
  OK  us_vix_log_ret
  OK  us_house_price_yoy
  OK  us_indprod_yoy
  OK  us_oil_yoy
  OK  us_reer_diff

df_model_usq: (144, 27) — all columns retained


In [8]:
DERIVED_USQ = [
    'us_house_price_yoy',
    'us_indprod_yoy',
    'us_oil_yoy',
    'us_reer_diff',
]

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[v.replace('us_', '').replace('_', ' ').title()
                    for v in DERIVED_USQ],
    horizontal_spacing=0.06)

for i, var in enumerate(DERIVED_USQ):
    fig.add_trace(go.Scatter(
        x=df_model_usq.index, y=df_model_usq[var],
        mode='lines', line=dict(color=BLUE, width=1.5),
        showlegend=False), row=1, col=i+1)
    fig.add_hline(y=df_model_usq[var].mean(), line_dash='dot',
                  line_color=NAVY, line_width=0.8, row=1, col=i+1)
    for s, e, _ in RECESSIONS_USQ:
        fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.25,
                      layer='below', line_width=0, row=1, col=i+1)

fig.update_xaxes(tickangle=45, tickfont=dict(size=8), nticks=5)
fig.update_yaxes(tickfont=dict(size=8), nticks=5)
fig.update_annotations(font_size=10)
fig.update_layout(
    title=dict(
        text='Figure 2a.1 — Derived Variables: US Quarterly | dotted=mean | grey=recessions',
        font=dict(size=12, color=NAVY)),
    template=TEMPLATE,
    height=300,
    margin=dict(t=80, b=50, l=50, r=30))
fig.show()

### 2b — US Monthly

In [9]:
df_usm['us_house_price_yoy'] = df_usm['us_house_price_idx'].pct_change(12) * 100
df_usm['us_indprod_yoy']     = df_usm['us_industrial_production'].pct_change(12) * 100
df_usm['us_oil_yoy']         = df_usm['us_oil_price'].pct_change(12) * 100
df_usm['us_reer_interp']     = df_usm['us_reer'].interpolate(method='linear', limit_direction='both')
df_usm['us_reer_diff']       = df_usm['us_reer_interp'].diff()

print('Derived columns added to df_usm:')
for c in ['us_house_price_yoy', 'us_indprod_yoy', 'us_oil_yoy', 'us_reer_interp', 'us_reer_diff']:
    print(f'  {c:30s}  non-null: {df_usm[c].notna().sum()}')
print(f'Total columns now: {df_usm.shape[1]}  (was {raw_usm.shape[1]})')

Derived columns added to df_usm:
  us_house_price_yoy              non-null: 422
  us_indprod_yoy                  non-null: 422
  us_oil_yoy                      non-null: 422
  us_reer_interp                  non-null: 434
  us_reer_diff                    non-null: 433
Total columns now: 27  (was 22)


In [10]:
FEATURES_USM = [
    'us_gdp_yoy_growth',
    'us_unemployment',
    'us_cpi',
    'us_consumer_confidence',
    'us_bond_yield_10y',
    'us_credit_mom_growth',
    'us_sp500_log_ret',
    'us_vix_log_ret',
    'us_house_price_yoy',
    'us_indprod_yoy',
    'us_oil_yoy',
    'us_reer_diff',
]

EXCL_USM = {k: v for k, v in {
    'us_real_gdp':              'Raw data — replaced by us_gdp_yoy_growth',
    'us_credit':                'Raw data — replaced by us_credit_mom_growth',
    'us_sp500_close':           'Raw data — replaced by us_sp500_log_ret',
    'us_house_price_idx':       'Raw data — replaced by us_house_price_yoy',
    'us_industrial_production': 'Raw data — replaced by us_indprod_yoy',
    'us_oil_price':             'Raw data — replaced by us_oil_yoy',
    'us_vix':                   'Raw data — replaced by us_vix_log_ret',
    'us_reer':                  'Raw data — replaced by us_reer_diff',
    'us_credit_yoy_growth':     'Transformation — us_credit_mom_growth preferred at monthly freq',
    'us_reer_log_ret':          'Transformation — us_reer_diff preferred for consistency',
    'us_oil_log_ret':           'Transformation — us_oil_yoy preferred for interpretability',
    'us_bond_yield_1y':         'Redundant — collinear with us_bond_yield_10y',
    'us_term_spread':           'Redundant — fully determined by bond_yield_10y and bond_yield_1y',
    'us_reer_interp':           'Intermediate — derivation step for us_reer_diff only',
}.items() if k in df_usm.columns}

df_model_usm = df_usm.copy()

print(f'FEATURES_USM ({len(FEATURES_USM)} in pipeline):')
for f in FEATURES_USM:
    print(f'  {"OK" if f in df_model_usm.columns else "MISSING"}  {f}')
print('Key difference vs US_Q: us_credit_mom_growth replaces us_credit_qoq_growth')
print(f'\ndf_model_usm: {df_model_usm.shape} — all columns retained')

FEATURES_USM (12 in pipeline):
  OK  us_gdp_yoy_growth
  OK  us_unemployment
  OK  us_cpi
  OK  us_consumer_confidence
  OK  us_bond_yield_10y
  OK  us_credit_mom_growth
  OK  us_sp500_log_ret
  OK  us_vix_log_ret
  OK  us_house_price_yoy
  OK  us_indprod_yoy
  OK  us_oil_yoy
  OK  us_reer_diff
Key difference vs US_Q: us_credit_mom_growth replaces us_credit_qoq_growth

df_model_usm: (434, 27) — all columns retained


In [11]:
DERIVED_USM = [
    'us_house_price_yoy',
    'us_indprod_yoy',
    'us_oil_yoy',
    'us_reer_diff',
]

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[v.replace('us_', '').replace('_', ' ').title()
                    for v in DERIVED_USM],
    horizontal_spacing=0.06)

for i, var in enumerate(DERIVED_USM):
    fig.add_trace(go.Scatter(
        x=df_model_usm.index, y=df_model_usm[var],
        mode='lines', line=dict(color=BLUE, width=1.2),
        showlegend=False), row=1, col=i+1)
    fig.add_hline(y=df_model_usm[var].mean(), line_dash='dot',
                  line_color=NAVY, line_width=0.8, row=1, col=i+1)
    for s, e, _ in RECESSIONS_USM:
        fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.25,
                      layer='below', line_width=0, row=1, col=i+1)

fig.update_xaxes(tickangle=45, tickfont=dict(size=8), nticks=5)
fig.update_yaxes(tickfont=dict(size=8), nticks=5)
fig.update_annotations(font_size=10)
fig.update_layout(
    title=dict(
        text='Figure 2b.1 — Derived Variables: US Monthly | dotted=mean | grey=recessions',
        font=dict(size=12, color=NAVY)),
    template=TEMPLATE,
    height=300,
    margin=dict(t=80, b=50, l=50, r=30))
fig.show()

### 2c — UK Quarterly

In [12]:
df_ukq['uk_house_price_yoy'] = df_ukq['uk_house_price_idx'].pct_change(4) * 100
df_ukq['uk_indprod_yoy']     = df_ukq['uk_industrial_production'].pct_change(4) * 100
df_ukq['uk_oil_yoy']         = df_ukq['uk_oil_price'].pct_change(4) * 100
df_ukq['uk_reer_interp']     = df_ukq['uk_reer'].interpolate(method='linear', limit_direction='both')
df_ukq['uk_reer_diff']       = df_ukq['uk_reer_interp'].diff()

print('Derived columns added to df_ukq:')
for c in ['uk_house_price_yoy', 'uk_indprod_yoy', 'uk_oil_yoy', 'uk_reer_interp', 'uk_reer_diff']:
    print(f'  {c:30s}  non-null: {df_ukq[c].notna().sum()}')
print(f'Total columns now: {df_ukq.shape[1]}  (was {raw_ukq.shape[1]})')

Derived columns added to df_ukq:
  uk_house_price_yoy              non-null: 140
  uk_indprod_yoy                  non-null: 140
  uk_oil_yoy                      non-null: 140
  uk_reer_interp                  non-null: 144
  uk_reer_diff                    non-null: 143
Total columns now: 20  (was 15)


In [13]:
FEATURES_UKQ = [
    'uk_gdp_yoy_growth',
    'uk_unemployment',
    'uk_cpi',
    'uk_consumer_confidence',
    'uk_bond_yield_10y',
    'uk_credit_qoq_growth',
    'uk_ftse_qoq_growth',
    'uk_house_price_yoy',
    'uk_indprod_yoy',
    'uk_oil_yoy',
    'uk_reer_diff',
]

EXCL_UKQ = {
    'uk_real_gdp':              'Raw data — replaced by uk_gdp_yoy_growth',
    'uk_credit':                'Raw data — replaced by uk_credit_qoq_growth',
    'uk_ftse_close':            'Raw data — replaced by uk_ftse_qoq_growth',
    'uk_house_price_idx':       'Raw data — replaced by uk_house_price_yoy',
    'uk_industrial_production': 'Raw data — replaced by uk_indprod_yoy',
    'uk_oil_price':             'Raw data — replaced by uk_oil_yoy',
    'uk_reer':                  'Raw data — replaced by uk_reer_diff',
    'uk_credit_yoy_growth':     'Transformation — uk_credit_qoq_growth preferred at quarterly freq',
    'uk_reer_interp':           'Intermediate — derivation step for uk_reer_diff only',
}

df_model_ukq = df_ukq.copy()

print(f'FEATURES_UKQ ({len(FEATURES_UKQ)} in pipeline):')
for f in FEATURES_UKQ:
    print(f'  {"OK" if f in df_model_ukq.columns else "MISSING"}  {f}')
print('No VIX for UK — uk_ftse_qoq_growth is market proxy')
print('uk_indprod_yoy available only to ~2017 — limited forward-looking usefulness')
print(f'\ndf_model_ukq: {df_model_ukq.shape} — all columns retained')

FEATURES_UKQ (11 in pipeline):
  OK  uk_gdp_yoy_growth
  OK  uk_unemployment
  OK  uk_cpi
  OK  uk_consumer_confidence
  OK  uk_bond_yield_10y
  OK  uk_credit_qoq_growth
  OK  uk_ftse_qoq_growth
  OK  uk_house_price_yoy
  OK  uk_indprod_yoy
  OK  uk_oil_yoy
  OK  uk_reer_diff
No VIX for UK — uk_ftse_qoq_growth is market proxy
uk_indprod_yoy available only to ~2017 — limited forward-looking usefulness

df_model_ukq: (144, 20) — all columns retained


In [14]:
DERIVED_UKQ = [
    'uk_house_price_yoy',
    'uk_indprod_yoy',
    'uk_oil_yoy',
    'uk_reer_diff',
]

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[v.replace('uk_', '').replace('_', ' ').title()
                    for v in DERIVED_UKQ],
    horizontal_spacing=0.06)

for i, var in enumerate(DERIVED_UKQ):
    fig.add_trace(go.Scatter(
        x=df_model_ukq.index, y=df_model_ukq[var],
        mode='lines', line=dict(color=BLUE, width=1.5),
        showlegend=False), row=1, col=i+1)
    fig.add_hline(y=df_model_ukq[var].mean(), line_dash='dot',
                  line_color=NAVY, line_width=0.8, row=1, col=i+1)
    for s, e, _ in RECESSIONS_UKQ:
        fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.25,
                      layer='below', line_width=0, row=1, col=i+1)

fig.update_xaxes(tickangle=45, tickfont=dict(size=8), nticks=5)
fig.update_yaxes(tickfont=dict(size=8), nticks=5)
fig.update_annotations(font_size=10)
fig.update_layout(
    title=dict(
        text='Figure 2c.1 — Derived Variables: UK Quarterly | dotted=mean | grey=recessions<br>'
             '<sup>uk_indprod_yoy available to 2017 only</sup>',
        font=dict(size=12, color=NAVY)),
    template=TEMPLATE,
    height=300,
    margin=dict(t=100, b=50, l=50, r=30))
fig.show()

## **3: Stationarity Testing (ADF + KPSS)**
___
ADF (H₀: unit root) and KPSS (H₀: stationary) are run together — agreement between both
gives strong confirmation. Bond yield uses constant+trend spec (`ct`) due to its secular
downward trend (Bank & Eder 2021, p. 4).

Non-stationary variables are first-differenced — the `_d1` column is added to the
dataframe and the original is never removed.


In [15]:
def run_stat(series, name, reg='c'):
    s = series.dropna()
    adf_s, adf_p, *_ = adfuller(s, autolag='AIC', regression=reg)
    kpss_s, kpss_p, *_ = kpss(s, regression=('ct' if reg == 'ct' else 'c'), nlags='auto')
    ar, kr = adf_p < 0.05, kpss_p < 0.05
    if ar and not kr:       dec = 'STATIONARY'
    elif not ar and kr:     dec = 'NON-STATIONARY'
    elif not ar and not kr: dec = 'INCONCLUSIVE'
    else:                   dec = 'CONFLICTING'
    return {'Variable': name, 'N': len(s),
            'ADF stat': round(adf_s, 3), 'ADF p': round(adf_p, 3),
            'KPSS stat': round(kpss_s, 3), 'KPSS p': round(kpss_p, 3),
            'Spec': reg, 'Decision': dec}

def stat_block(df_m, features, target, prefix, label):
    bond = prefix + 'bond_yield_10y'
    results = []
    for var in features:
        results.append(run_stat(df_m[var].dropna(), var, 'ct' if var == bond else 'c'))
    if target and target in df_m.columns:
        results.append(run_stat(df_m[target].dropna(), target, 'c'))
    stat_df = pd.DataFrame(results)
    display(stat_df.style
            .map(c_stat, subset=['Decision'])
            .map(c_p, subset=['ADF p'])
            .format({'ADF stat': '{:.3f}', 'ADF p': '{:.3f}',
                     'KPSS stat': '{:.3f}', 'KPSS p': '{:.3f}'})
            .set_caption(f'Table 3.1 — ADF+KPSS: {label}')
            .hide(axis='index'))
    short = [v.replace(prefix, '').replace('_', ' ').title() for v in stat_df['Variable']]
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=['ADF p-values (green = reject H₀ → stationary)',
                        'KPSS p-values (green = fail to reject H₀ → stationary)'],
        horizontal_spacing=0.12)
    fig.add_trace(go.Bar(x=stat_df['ADF p'], y=short, orientation='h',
        marker_color=[GREEN if p < 0.05 else RED for p in stat_df['ADF p']],
        hovertemplate='%{y}<br>ADF p=%{x:.3f}<extra></extra>'), row=1, col=1)
    fig.add_vline(x=0.05, line_dash='dash', line_color=NAVY, line_width=1.5, row=1, col=1)
    fig.add_trace(go.Bar(x=stat_df['KPSS p'], y=short, orientation='h',
        marker_color=[GREEN if p > 0.05 else RED for p in stat_df['KPSS p']],
        hovertemplate='%{y}<br>KPSS p=%{x:.3f}<extra></extra>'), row=1, col=2)
    fig.add_vline(x=0.05, line_dash='dash', line_color=NAVY, line_width=1.5, row=1, col=2)
    fig.update_layout(
        title=dict(text=f'Figure 3.1 — Stationarity p-values: {label}', font=dict(size=12, color=NAVY)),
        template=TEMPLATE,
        height=350,
        showlegend=False,
        margin=dict(t=60, b=40, l=140, r=40))
    fig.update_xaxes(title_text='p-value', range=[0, 1], tickfont=dict(size=8))
    fig.update_yaxes(tickfont=dict(size=8))
    fig.show()
    return stat_df

def handle_nonstat(df_m, stat_df, features, label):
    non_stat  = stat_df[stat_df['Decision'] == 'NON-STATIONARY']['Variable'].tolist()
    feat_stat = features.copy()
    retest    = []
    for var in non_stat:
        if var in feat_stat:
            new = var + '_d1'
            df_m[new] = df_m[var].diff()
            feat_stat[feat_stat.index(var)] = new
            retest.append(run_stat(df_m[new].dropna(), new))
            print(f'  {var}  →  {new}  (column added, original retained)')
    if retest:
        rt = pd.DataFrame(retest)
        display(rt.style
                .map(c_stat, subset=['Decision'])
                .format({'ADF p': '{:.3f}', 'KPSS p': '{:.3f}'})
                .set_caption(f'Retest after first differencing: {label}')
                .hide(axis='index'))
    elif non_stat:
        print('All non-stationary variables handled.')
    else:
        print(f'No non-stationary variables found in {label}.')
    return feat_stat

print('Stationarity functions defined.')


Stationarity functions defined.


### 3a — US Quarterly

In [16]:
stat_df_usq = stat_block(df_model_usq, FEATURES_USQ, TARGET_USQ, PREFIX_USQ, 'US Quarterly')
print('Handling non-stationary:')
FEATURES_STAT_USQ = handle_nonstat(df_model_usq, stat_df_usq, FEATURES_USQ, 'US Quarterly')
print(f'FEATURES_STAT_USQ ({len(FEATURES_STAT_USQ)}): {FEATURES_STAT_USQ}')

Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_gdp_yoy_growth,144,-2.877,0.048,0.184,0.100,c,STATIONARY
us_unemployment,144,-3.180,0.021,0.174,0.100,c,STATIONARY
us_cpi,143,-3.156,0.023,0.166,0.100,c,STATIONARY
us_consumer_confidence,144,-5.426,0.000,0.042,0.100,c,STATIONARY
us_bond_yield_10y,144,-2.137,0.525,0.257,0.010,ct,NON-STATIONARY
us_credit_qoq_growth,143,-3.083,0.028,0.296,0.100,c,STATIONARY
us_sp500_log_ret,144,-11.794,0.000,0.106,0.100,c,STATIONARY
us_vix_log_ret,143,-11.915,0.000,0.040,0.100,c,STATIONARY
us_house_price_yoy,140,-2.688,0.076,0.200,0.100,c,INCONCLUSIVE
us_indprod_yoy,140,-3.241,0.018,0.400,0.077,c,STATIONARY


Handling non-stationary:
  us_bond_yield_10y  →  us_bond_yield_10y_d1  (column added, original retained)


Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_bond_yield_10y_d1,143,-9.333000,0.000,0.198000,0.100,c,STATIONARY


FEATURES_STAT_USQ (12): ['us_gdp_yoy_growth', 'us_unemployment', 'us_cpi', 'us_consumer_confidence', 'us_bond_yield_10y_d1', 'us_credit_qoq_growth', 'us_sp500_log_ret', 'us_vix_log_ret', 'us_house_price_yoy', 'us_indprod_yoy', 'us_oil_yoy', 'us_reer_diff']


### 3b — US Monthly

In [17]:
stat_df_usm = stat_block(df_model_usm, FEATURES_USM, TARGET_USM, PREFIX_USM, 'US Monthly')
print('Handling non-stationary:')
FEATURES_STAT_USM = handle_nonstat(df_model_usm, stat_df_usm, FEATURES_USM, 'US Monthly')
print(f'FEATURES_STAT_USM ({len(FEATURES_STAT_USM)}): {FEATURES_STAT_USM}')
print('Note: ~400+ obs increases test power vs quarterly — borderline cases more decisive.')

Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_gdp_yoy_growth,432,-4.260,0.001,0.218,0.100,c,STATIONARY
us_unemployment,432,-2.779,0.061,0.300,0.100,c,INCONCLUSIVE
us_cpi,429,-4.268,0.001,0.234,0.100,c,STATIONARY
us_consumer_confidence,434,-5.156,0.000,0.048,0.100,c,STATIONARY
us_bond_yield_10y,434,-2.349,0.407,0.442,0.010,ct,NON-STATIONARY
us_credit_mom_growth,429,-3.101,0.027,0.571,0.026,c,CONFLICTING
us_sp500_log_ret,434,-20.789,0.000,0.120,0.100,c,STATIONARY
us_vix_log_ret,433,-11.758,0.000,0.041,0.100,c,STATIONARY
us_house_price_yoy,422,-2.340,0.159,0.294,0.100,c,INCONCLUSIVE
us_indprod_yoy,422,-3.472,0.009,0.485,0.045,c,CONFLICTING


Handling non-stationary:
  us_bond_yield_10y  →  us_bond_yield_10y_d1  (column added, original retained)


Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_bond_yield_10y_d1,433,-10.958000,0.000,0.178000,0.100,c,STATIONARY


FEATURES_STAT_USM (12): ['us_gdp_yoy_growth', 'us_unemployment', 'us_cpi', 'us_consumer_confidence', 'us_bond_yield_10y_d1', 'us_credit_mom_growth', 'us_sp500_log_ret', 'us_vix_log_ret', 'us_house_price_yoy', 'us_indprod_yoy', 'us_oil_yoy', 'us_reer_diff']
Note: ~400+ obs increases test power vs quarterly — borderline cases more decisive.


### 3c — UK Quarterly

In [18]:
stat_df_ukq = stat_block(df_model_ukq, FEATURES_UKQ, None, PREFIX_UKQ, 'UK Quarterly')
print('Handling non-stationary:')
FEATURES_STAT_UKQ = handle_nonstat(df_model_ukq, stat_df_ukq, FEATURES_UKQ, 'UK Quarterly')
print(f'FEATURES_STAT_UKQ ({len(FEATURES_STAT_UKQ)}): {FEATURES_STAT_UKQ}')

Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
uk_gdp_yoy_growth,144,-5.178,0.000,0.088,0.100,c,STATIONARY
uk_unemployment,144,-3.512,0.008,0.784,0.010,c,CONFLICTING
uk_cpi,144,-3.220,0.019,0.202,0.100,c,STATIONARY
uk_consumer_confidence,144,-3.815,0.003,0.174,0.100,c,STATIONARY
uk_bond_yield_10y,144,-1.846,0.682,0.252,0.010,ct,NON-STATIONARY
uk_credit_qoq_growth,143,-11.084,0.000,0.341,0.100,c,STATIONARY
uk_ftse_qoq_growth,144,-12.964,0.000,0.069,0.100,c,STATIONARY
uk_house_price_yoy,140,-2.046,0.267,0.228,0.100,c,INCONCLUSIVE
uk_indprod_yoy,140,-2.369,0.151,0.161,0.100,c,INCONCLUSIVE
uk_oil_yoy,140,-4.644,0.000,0.075,0.100,c,STATIONARY


Handling non-stationary:
  uk_bond_yield_10y  →  uk_bond_yield_10y_d1  (column added, original retained)


Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
uk_bond_yield_10y_d1,143,-8.518000,0.000,0.552000,0.030,c,CONFLICTING


FEATURES_STAT_UKQ (11): ['uk_gdp_yoy_growth', 'uk_unemployment', 'uk_cpi', 'uk_consumer_confidence', 'uk_bond_yield_10y_d1', 'uk_credit_qoq_growth', 'uk_ftse_qoq_growth', 'uk_house_price_yoy', 'uk_indprod_yoy', 'uk_oil_yoy', 'uk_reer_diff']


## **4: Distributional Analysis & Outlier Detection**
___
Stress-period outliers are retained — they carry exactly the signal needed for IFRS 9
scenario analysis (Bank & Eder 2021, Section 10). Plots use a non-null mask for clean
visualisation; no rows are removed from the dataframes.


In [19]:
def dist_table(df_m, features, prefix, target, label):
    vp = features + ([target] if target and target in df_m.columns else [])
    rows = []
    for var in vp:
        s   = df_m[var].dropna()
        iqr = s.quantile(0.75) - s.quantile(0.25)
        out = int(((s < s.quantile(0.25) - 3*iqr) | (s > s.quantile(0.75) + 3*iqr)).sum())
        rows.append({
            'Variable':         var.replace(prefix, '').replace('_', ' ').title(),
            'N':                len(s),
            'Mean':             round(s.mean(), 3),
            'Median':           round(s.median(), 3),
            'Std':              round(s.std(), 3),
            'Min':              round(s.min(), 3),
            'Max':              round(s.max(), 3),
            'Skewness':         round(s.skew(), 3),
            'Kurtosis':         round(s.kurtosis(), 3),
            'Outliers (3xIQR)': out,
        })
    d = pd.DataFrame(rows)
    display(d.style
            .map(c_skew, subset=['Skewness'])
            .map(c_kurt, subset=['Kurtosis'])
            .map(c_out,  subset=['Outliers (3xIQR)'])
            .format({c: '{:.3f}' for c in ['Mean', 'Median', 'Std', 'Min', 'Max', 'Skewness', 'Kurtosis']})
            .set_caption(f'Table 4.1 — Distributions: {label}')
            .hide(axis='index'))

def hist_kde(df_m, features, prefix, target, label):
    vp    = features + ([target] if target and target in df_m.columns else [])
    ncols = 4
    nrows = int(np.ceil(len(vp) / ncols))
    fig   = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[v.replace(prefix, '').replace('_', ' ').title() for v in vp],
        vertical_spacing=min(0.06, 0.9 / nrows),
        horizontal_spacing=0.06)
    for i, var in enumerate(vp):
        r, c = i // ncols + 1, i % ncols + 1
        s    = df_m[var].dropna()
        col  = RED if var == target else BLUE
        fig.add_trace(go.Histogram(
            x=s, nbinsx=30, marker_color=col, opacity=0.55,
            histnorm='probability density', showlegend=False), row=r, col=c)
        kx = np.linspace(s.min(), s.max(), 200)
        fig.add_trace(go.Scatter(
            x=kx, y=stats.gaussian_kde(s)(kx), mode='lines',
            line=dict(color=NAVY, width=1.5), showlegend=False), row=r, col=c)
        fig.add_trace(go.Scatter(
            x=kx, y=stats.norm.pdf(kx, s.mean(), s.std()), mode='lines',
            line=dict(color=GREY, width=1.2, dash='dash'), showlegend=False), row=r, col=c)
        xr = 'x domain' if i == 0 else f'x{i+1} domain'
        yr = 'y domain' if i == 0 else f'y{i+1} domain'
        ac = RED if abs(s.skew()) > 2 else (AMBER if abs(s.skew()) > 1 else GREEN)
        fig.add_annotation(
            xref=xr, yref=yr, x=0.97, y=0.95,
            text=f'skew={s.skew():.2f}<br>kurt={s.kurtosis():.2f}',
            showarrow=False, font=dict(size=7, color=ac),
            bgcolor='white', bordercolor=ac, borderwidth=0.8, align='right')
    fig.update_xaxes(tickfont=dict(size=7), nticks=4)
    fig.update_yaxes(tickfont=dict(size=7), nticks=3, showticklabels=False)
    fig.update_annotations(font_size=9)
    fig.update_layout(
        title=dict(
            text=f'Figure 4.1 — Distributions: {label} | navy=KDE | grey=normal | red=target',
            font=dict(size=12, color=NAVY)),
        template=TEMPLATE,
        height=nrows * 200,
        margin=dict(t=80, b=30, l=30, r=30))
    fig.show()

def rec_stress(df_m, features, prefix, recessions, label):
    in_rec = pd.Series(False, index=df_m.index)
    for s, e, _ in recessions:
        in_rec |= ((df_m.index >= s) & (df_m.index <= e))
    st = pd.DataFrame({
        'During recession':  df_m[features].loc[in_rec].median(),
        'Outside recession': df_m[features].loc[~in_rec].median(),
    })
    st['Difference'] = st['During recession'] - st['Outside recession']
    def exp(v):
        vl = v.lower()
        for k in ['vix', 'cpi']:
            if k in vl: return 'Expect positive'
        for k in ['gdp', 'sp500', 'ftse', 'indprod', 'house_price', 'credit',
                  'oil', 'consumer_confidence', 'bond_yield']:
            if k in vl: return 'Expect negative'
        return 'Ambiguous'
    st['Expected']  = [exp(v) for v in st.index]
    st['Confirmed'] = [
        'Yes'  if (st.loc[v, 'Difference'] > 0 and 'positive' in st.loc[v, 'Expected'])
                  or (st.loc[v, 'Difference'] < 0 and 'negative' in st.loc[v, 'Expected'])
        else ('N/A' if 'Ambiguous' in st.loc[v, 'Expected'] else 'Check')
        for v in st.index]
    def c_diff(v):
        if v < -0.5: return 'background-color:#EBF5FB'
        if v >  0.5: return 'background-color:#FDEBD0'
        return ''
    display(st.style
            .map(c_confirmed, subset=['Confirmed'])
            .map(c_diff,      subset=['Difference'])
            .format({'During recession': '{:.3f}', 'Outside recession': '{:.3f}', 'Difference': '{:+.3f}'})
            .set_caption(f'Table 4.2 — Recession Stress: {label}'))

def outlier_summ(df_m, features, prefix, label):
    sp = pd.Series(
        ((df_m.index >= '2008-01-01') & (df_m.index <= '2009-12-31')) |
        ((df_m.index >= '2020-01-01') & (df_m.index <= '2020-12-31')) |
        ((df_m.index >= '2021-01-01') & (df_m.index <= '2023-12-31')),
        index=df_m.index)
    rows = []
    for var in features:
        s   = df_m[var].dropna()
        iqr = s.quantile(0.75) - s.quantile(0.25)
        iso = ((s < s.quantile(0.25) - 3*iqr) | (s > s.quantile(0.75) + 3*iqr))
        spa = sp.reindex(iso.index, fill_value=False)
        tot = int(iso.sum()); ins = int((iso & spa).sum()); out = tot - ins
        rows.append({
            'Variable':       var.replace(prefix, '').replace('_', ' ').title(),
            'Total':          tot,
            'In stress':      ins,
            'Outside stress': out,
            'Action':         'Retain — stress signal' if ins > 0 else ('Investigate' if out > 0 else 'None'),
        })
    display(pd.DataFrame(rows).style
            .map(c_action, subset=['Action'])
            .set_caption(f'Table 4.3 — Outliers: {label}')
            .hide(axis='index'))

print('Distributional functions defined.')

Distributional functions defined.


### 4a — US Quarterly

In [20]:
dist_table(df_model_usq, FEATURES_STAT_USQ, PREFIX_USQ, TARGET_USQ, 'US Quarterly')
hist_kde(df_model_usq,   FEATURES_STAT_USQ, PREFIX_USQ, TARGET_USQ, 'US Quarterly')
rec_stress(df_model_usq, FEATURES_STAT_USQ, PREFIX_USQ, RECESSIONS_USQ, 'US Quarterly')
outlier_summ(df_model_usq, FEATURES_STAT_USQ, PREFIX_USQ, 'US Quarterly')

Variable,N,Mean,Median,Std,Min,Max,Skewness,Kurtosis,Outliers (3xIQR)
Gdp Yoy Growth,144,2.501,2.605,1.998,-7.399,12.386,-0.682,8.392,5
Unemployment,144,5.674,5.333,1.734,3.533,13.000,1.226,1.649,0
Cpi,143,2.700,2.641,1.581,-1.623,8.636,1.063,2.998,3
Consumer Confidence,144,99.784,99.905,1.232,94.780,102.192,-1.087,2.285,0
Bond Yield 10Y D1,143,-0.031,0.000,0.455,-1.270,1.010,-0.089,-0.213,0
Credit Qoq Growth,143,1.269,1.229,0.870,-0.938,2.994,-0.268,-0.244,0
Sp500 Log Ret,144,0.021,0.031,0.079,-0.256,0.190,-0.971,1.362,1
Vix Log Ret,143,-0.002,-0.028,0.299,-0.617,1.357,1.099,2.985,1
House Price Yoy,140,2.155,3.308,4.586,-12.567,13.026,-0.733,1.110,0
Indprod Yoy,140,1.479,2.332,4.005,-15.282,8.958,-1.521,3.919,2


,During recession,Outside recession,Difference,Expected,Confirmed
us_gdp_yoy_growth,0.603,2.791,-2.188,Expect negative,Yes
us_unemployment,5.500,5.300,+0.200,Ambiguous,N/A
us_cpi,3.393,2.580,+0.813,Expect positive,Yes
us_consumer_confidence,98.078,99.948,-1.870,Expect negative,Yes
us_bond_yield_10y_d1,-0.350,0.000,-0.350,Expect negative,Yes
us_credit_qoq_growth,1.038,1.251,-0.213,Expect negative,Yes
us_sp500_log_ret,-0.093,0.036,-0.129,Expect negative,Yes
us_vix_log_ret,0.065,-0.030,+0.094,Expect positive,Yes
us_house_price_yoy,-5.965,3.413,-9.378,Expect negative,Yes
us_indprod_yoy,-4.550,2.531,-7.081,Expect negative,Yes


Variable,Total,In stress,Outside stress,Action
Gdp Yoy Growth,5,5,0,Retain — stress signal
Unemployment,0,0,0,None
Cpi,3,3,0,Retain — stress signal
Consumer Confidence,0,0,0,None
Bond Yield 10Y D1,0,0,0,None
Credit Qoq Growth,0,0,0,None
Sp500 Log Ret,1,1,0,Retain — stress signal
Vix Log Ret,1,1,0,Retain — stress signal
House Price Yoy,0,0,0,None
Indprod Yoy,2,2,0,Retain — stress signal


### 4b — US Monthly

In [21]:
dist_table(df_model_usm, FEATURES_STAT_USM, PREFIX_USM, TARGET_USM, 'US Monthly')
hist_kde(df_model_usm,   FEATURES_STAT_USM, PREFIX_USM, TARGET_USM, 'US Monthly')
rec_stress(df_model_usm, FEATURES_STAT_USM, PREFIX_USM, RECESSIONS_USM, 'US Monthly')
outlier_summ(df_model_usm, FEATURES_STAT_USM, PREFIX_USM, 'US Monthly')


Variable,N,Mean,Median,Std,Min,Max,Skewness,Kurtosis,Outliers (3xIQR)
Gdp Yoy Growth,432,2.502,2.603,1.960,-7.399,12.386,-0.798,6.251,15
Unemployment,432,5.676,5.333,1.719,2.676,13.000,1.132,1.188,0
Cpi,429,2.704,2.621,1.576,-1.845,8.659,1.054,2.875,8
Consumer Confidence,434,99.777,99.873,1.272,92.876,102.192,-1.391,3.975,2
Bond Yield 10Y D1,433,-0.009,-0.020,0.218,-1.110,0.650,-0.093,1.437,1
Credit Mom Growth,429,0.420,0.416,0.292,-0.425,1.057,-0.249,-0.226,0
Sp500 Log Ret,434,0.007,0.012,0.043,-0.186,0.119,-0.731,1.438,1
Vix Log Ret,433,-0.001,-0.011,0.199,-0.614,0.853,0.417,1.253,1
House Price Yoy,422,2.131,3.318,4.577,-12.567,13.348,-0.716,1.018,0
Indprod Yoy,422,1.485,2.284,4.186,-17.318,16.552,-1.387,4.582,10


,During recession,Outside recession,Difference,Expected,Confirmed
us_gdp_yoy_growth,0.776,2.697,-1.922,Expect negative,Yes
us_unemployment,5.531,5.326,+0.205,Ambiguous,N/A
us_cpi,3.435,2.569,+0.865,Expect positive,Yes
us_consumer_confidence,98.129,99.933,-1.804,Expect negative,Yes
us_bond_yield_10y_d1,-0.050,-0.020,-0.030,Expect negative,Yes
us_credit_mom_growth,0.355,0.438,-0.083,Expect negative,Yes
us_sp500_log_ret,-0.006,0.013,-0.019,Expect negative,Yes
us_vix_log_ret,-0.016,-0.010,-0.006,Expect positive,Check
us_house_price_yoy,-6.517,3.347,-9.863,Expect negative,Yes
us_indprod_yoy,-3.683,2.484,-6.167,Expect negative,Yes


Variable,Total,In stress,Outside stress,Action
Gdp Yoy Growth,15,15,0,Retain — stress signal
Unemployment,0,0,0,None
Cpi,8,8,0,Retain — stress signal
Consumer Confidence,2,2,0,Retain — stress signal
Bond Yield 10Y D1,1,1,0,Retain — stress signal
Credit Mom Growth,0,0,0,None
Sp500 Log Ret,1,1,0,Retain — stress signal
Vix Log Ret,1,0,1,Investigate
House Price Yoy,0,0,0,None
Indprod Yoy,10,10,0,Retain — stress signal


### 4c — UK Quarterly

In [22]:
dist_table(df_model_ukq, FEATURES_STAT_UKQ, PREFIX_UKQ, None, 'UK Quarterly')
hist_kde(df_model_ukq,   FEATURES_STAT_UKQ, PREFIX_UKQ, None, 'UK Quarterly')
rec_stress(df_model_ukq, FEATURES_STAT_UKQ, PREFIX_UKQ, RECESSIONS_UKQ, 'UK Quarterly')
outlier_summ(df_model_ukq, FEATURES_STAT_UKQ, PREFIX_UKQ, 'UK Quarterly')


Variable,N,Mean,Median,Std,Min,Max,Skewness,Kurtosis,Outliers (3xIQR)
Gdp Yoy Growth,144,1.827,2.089,3.793,-21.671,25.478,-0.264,20.508,10
Unemployment,144,6.165,5.500,1.831,3.700,10.600,0.680,-0.603,0
Cpi,144,2.820,2.300,1.981,0.300,9.400,1.795,2.839,11
Consumer Confidence,144,99.964,100.257,1.613,94.034,103.272,-1.317,2.618,3
Bond Yield 10Y D1,143,-0.056,-0.074,0.427,-1.260,1.293,0.263,0.616,0
Credit Qoq Growth,143,1.426,1.313,4.786,-14.552,14.877,-0.339,0.952,0
Ftse Qoq Growth,144,1.246,1.911,7.159,-24.798,20.820,-0.618,1.366,1
House Price Yoy,140,2.486,2.155,7.081,-17.733,22.501,-0.060,0.456,0
Indprod Yoy,140,0.113,0.000,2.547,-9.928,6.158,-1.176,3.948,4
Oil Yoy,140,10.659,3.078,45.579,-78.139,327.744,2.691,16.087,1


,During recession,Outside recession,Difference,Expected,Confirmed
uk_gdp_yoy_growth,-1.423,2.225,-3.648,Expect negative,Yes
uk_unemployment,6.400,5.500,+0.900,Ambiguous,N/A
uk_cpi,3.300,2.300,+1.000,Expect positive,Yes
uk_consumer_confidence,97.125,100.313,-3.188,Expect negative,Yes
uk_bond_yield_10y_d1,-0.370,-0.050,-0.320,Expect negative,Yes
uk_credit_qoq_growth,0.941,1.327,-0.386,Expect negative,Yes
uk_ftse_qoq_growth,-9.552,2.076,-11.629,Expect negative,Yes
uk_house_price_yoy,-9.567,2.909,-12.475,Expect negative,Yes
uk_indprod_yoy,-2.131,0.000,-2.131,Expect negative,Yes
uk_oil_yoy,-38.329,5.533,-43.862,Expect negative,Yes


Variable,Total,In stress,Outside stress,Action
Gdp Yoy Growth,10,10,0,Retain — stress signal
Unemployment,0,0,0,None
Cpi,11,5,6,Retain — stress signal
Consumer Confidence,3,3,0,Retain — stress signal
Bond Yield 10Y D1,0,0,0,None
Credit Qoq Growth,0,0,0,None
Ftse Qoq Growth,1,1,0,Retain — stress signal
House Price Yoy,0,0,0,None
Indprod Yoy,4,4,0,Retain — stress signal
Oil Yoy,1,1,0,Retain — stress signal


## **5: Lag Structure & Cross-Correlation Analysis**
___
CCF identifies at which lag each variable has its strongest relationship with the target (Djeundje & Crook 2019; Bank & Eder 2021, Section 6.2). This is the most directly relevant EDA step for Stage 2 PD modelling — it determines which lag of each variable enters the PD regression.

**Variable-specific lag windows** are set based on economic transmission mechanisms rather than a single arbitrary global cutoff (Bank & Eder 2021, Section 10, p. 34):

| Variable | Max lag (Q) | Max lag (M) | Transmission rationale | Reference |
|----------|-------------|-------------|------------------------|-----------|
| GDP growth | 4Q | 12M | Economic slowdowns affect borrower income within 1 year | Djeundje & Crook (2019) |
| Industrial production | 4Q | 12M | Contemporaneous business cycle indicator | Djeundje & Crook (2019) |
| Consumer confidence | 4Q | 12M | Leading indicator — moves before actual defaults | Bank & Eder (2021) |
| Unemployment | 4Q | 12M | Lagging indicator but still within 1-year horizon | Djeundje & Crook (2019) |
| S&P 500 / FTSE returns | 4Q | 12M | Market stress leads credit stress rapidly | Bank & Eder (2021) |
| VIX log return | 4Q | 12M | Fast-moving volatility signal | Bank & Eder (2021) |
| REER diff | 4Q | 12M | Exchange rate pass-through is relatively fast | Djeundje & Crook (2019) |
| Bond yield | 4Q | 12M | Rate changes affect debt servicing within 1 year | Bank & Eder (2021) |
| House prices | 6Q | 18M | Price declines take 6–18 months to feed through to mortgage stress | Djeundje & Crook (2019) |
| Credit growth | 6Q | 18M | Leverage build-up transmits to default over 1–2 years | Djeundje & Crook (2019) |
| CPI | 6Q | 18M | Inflation erodes real income gradually over multiple quarters | Bank & Eder (2021) |

**Significance threshold:** 95% confidence interval computed as ±1.96/√N (Bartlett approximation). Variables whose |Peak CCF| exceeds this threshold within their lag window are flagged as significant.

**After this section, two versions of each variable coexist in the dataframe:**
- **Unlagged** (e.g. `us_unemployment`) → Stage 1 ARIMA/ML macro forecasting
- **Lagged** (e.g. `us_unemployment_L2`) → Stage 2 PD regression at optimal lag

**UK dataset:** No default target available — CCF run against `uk_gdp_yoy_growth` as reference. Significant predictors indicate GDP drivers, not directly default drivers.

**References:**
Djeundje, V.B. & Crook, J. (2019). Dynamic survival models with varying coefficients for credit risks. *European Journal of Operational Research*, 275(1), 319–333.

Bank, M. & Eder, B. (2021). A Review on the Probability of Default for IFRS 9. SSRN Working Paper 3981339.

In [23]:
# Variable-specific maximum lag windows
# Based on economic transmission mechanisms — Bank & Eder (2021), Djeundje & Crook (2019)
LAG_WINDOWS_Q = {
    'gdp_yoy_growth':    4,   # Fast — economic slowdown within 1 year
    'indprod_yoy':       4,   # Fast — contemporaneous business cycle
    'consumer_confidence': 4, # Fast — leading indicator
    'unemployment':      4,   # Moderate — lagging but within 1 year
    'sp500_log_ret':     4,   # Fast — market leads credit stress
    'ftse_qoq_growth':   4,   # Fast — market leads credit stress
    'vix_log_ret':       4,   # Fast — volatility signal
    'reer_diff':         4,   # Fast — exchange rate pass-through
    'bond_yield_10y':    4,   # Moderate — rate changes within 1 year
    'bond_yield_10y_d1': 4,   # Moderate — rate changes within 1 year
    'house_price_yoy':   6,   # Slow — mortgage stress takes 6-18 months
    'credit_qoq_growth': 6,   # Slow — leverage transmits over 1-2 years
    'credit_mom_growth': 6,   # Slow — leverage transmits over 1-2 years
    'cpi':               6,   # Slow — real income erosion over multiple quarters
}
LAG_WINDOWS_M = {k: v * 3 for k, v in LAG_WINDOWS_Q.items()}


def get_max_lag(var, prefix, unit):
    """Return variable-specific max lag based on transmission mechanism."""
    short = var.replace(prefix, '').replace('_d1', '')
    windows = LAG_WINDOWS_M if unit == 'M' else LAG_WINDOWS_Q
    for key, lag in windows.items():
        if key in short:
            return lag
    return 4 if unit == 'Q' else 12  # default


def ccf_block(df_m, ref_col, features, prefix, label, unit='Q'):
    # Build aligned dataset using the maximum possible lag window
    global_max = max(get_max_lag(v, prefix, unit) for v in features)
    aligned    = df_m[[ref_col] + [f for f in features
                                   if f in df_m.columns]].dropna()
    N    = len(aligned)
    CONF = 1.96 / np.sqrt(N)
    ref  = aligned[ref_col].values
    print(f'{label}: N={N}, 95% CI=±{CONF:.3f}')
    print(f'Lag windows: Q={LAG_WINDOWS_Q} | global max={global_max}')

    # Compute CCF up to variable-specific max lag
    ccf_res  = {}
    max_lags = {}
    for var in features:
        if var not in aligned.columns: continue
        ml  = get_max_lag(var, prefix, unit)
        x   = aligned[var].values
        lag0  = np.corrcoef(ref, x)[0, 1]
        lags1 = ccf(ref, x, nlags=ml, alpha=None)
        ccf_res[var]  = np.concatenate([[lag0], lags1])
        max_lags[var] = ml

    # ── Panel plot ────────────────────────────────────────────────────────────
    ncols = 4
    nrows = int(np.ceil(len(ccf_res) / ncols))
    fig   = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[v.replace(prefix, '').replace('_', ' ').title()
                        for v in ccf_res],
        vertical_spacing=min(0.12, 0.9 / nrows),
        horizontal_spacing=0.08)

    for i, (var, vals) in enumerate(ccf_res.items()):
        r, c      = i // ncols + 1, i % ncols + 1
        lags_arr  = np.arange(0, len(vals))
        bc   = [GREEN if v > 0 else RED for v in vals]
        brd  = ['black' if abs(v) > CONF else 'rgba(0,0,0,0)' for v in vals]
        bw   = [1.5 if abs(v) > CONF else 0 for v in vals]
        fig.add_trace(go.Bar(
            x=lags_arr, y=vals,
            marker_color=bc, marker_line_color=brd, marker_line_width=bw,
            showlegend=False,
            hovertemplate=f'Lag %{{x}}{unit}<br>CCF=%{{y:.3f}}<extra></extra>'),
            row=r, col=c)
        fig.add_hline(y= CONF, line_dash='dot', line_color=GREY,
                      line_width=1, row=r, col=c)
        fig.add_hline(y=-CONF, line_dash='dot', line_color=GREY,
                      line_width=1, row=r, col=c)
        fig.add_hline(y=0, line_color='black', line_width=0.5, row=r, col=c)
        pl = int(np.argmax(np.abs(vals))); pv = vals[pl]
        xr = 'x domain' if i == 0 else f'x{i+1} domain'
        yr = 'y domain' if i == 0 else f'y{i+1} domain'
        fig.add_annotation(
            xref=xr, yref=yr, x=0.97, y=0.95,
            text=f'peak lag {pl}{unit}<br>CCF={pv:.2f}<br>max={max_lags[var]}{unit}',
            showarrow=False, font=dict(size=7, color=NAVY),
            bgcolor='white', bordercolor=NAVY, borderwidth=0.8, align='right')

    ref_short = ref_col.replace(prefix, '').replace('_', ' ').title()
    fig.update_xaxes(tickfont=dict(size=7))
    fig.update_yaxes(tickfont=dict(size=7))
    fig.update_annotations(font_size=8)
    fig.update_layout(
        title=dict(
            text=(f'Figure 5.1 — CCF vs {ref_short}: {label} | '
                  f'black border=sig 95% | annotation shows max lag window per variable'),
            font=dict(size=12, color=NAVY)),
        template=TEMPLATE,
        height=nrows * 240,
        margin=dict(t=80, b=40, l=40, r=30))
    fig.show()

    # ── Lag summary table ─────────────────────────────────────────────────────
    rows = []
    for var, vals in ccf_res.items():
        pl = int(np.argmax(np.abs(vals))); pv = vals[pl]
        rows.append({
            'Variable':           var,
            'Max lag window':     f'{max_lags[var]}{unit}',
            'Peak CCF':           round(pv, 3),
            '|Peak CCF|':         round(abs(pv), 3),
            f'Peak lag ({unit})': pl,
            'Direction':          'Positive' if pv > 0 else 'Negative',
            'Significant 95%':    'Yes' if abs(pv) > CONF else 'No',
        })
    lag_df = (pd.DataFrame(rows)
              .sort_values('|Peak CCF|', ascending=False)
              .reset_index(drop=True))
    display(lag_df.style
            .map(c_sig, subset=['Significant 95%'])
            .map(c_dir, subset=['Direction'])
            .format({'Peak CCF': '{:.3f}', '|Peak CCF|': '{:.3f}'})
            .set_caption(f'Table 5.1 — Lag Summary: {label}')
            .hide(axis='index'))

    # ── Magnitude ranking chart ───────────────────────────────────────────────
    cols  = [GREEN if d == 'Positive' else RED for d in lag_df['Direction']]
    bords = ['black' if s == 'Yes' else GREY for s in lag_df['Significant 95%']]
    fig2  = go.Figure()
    fig2.add_trace(go.Bar(
        x=lag_df['|Peak CCF|'],
        y=lag_df['Variable'].str.replace(prefix, '').str.replace('_', ' ').str.title(),
        orientation='h',
        marker_color=cols,
        marker_line_color=bords,
        marker_line_width=1.5,
        text=[f'lag {k}{unit}  CCF={v:+.2f}  (max {w})'
              for k, v, w in zip(lag_df[f'Peak lag ({unit})'],
                                 lag_df['Peak CCF'],
                                 lag_df['Max lag window'])],
        textposition='outside',
        textfont=dict(size=8)))
    fig2.add_vline(x=CONF, line_dash='dash', line_color=NAVY, line_width=1.5,
                   annotation_text=f'95% CI={CONF:.2f}',
                   annotation_font_color=NAVY,
                   annotation_font_size=9)
    fig2.update_layout(
        title=dict(
            text=f'Figure 5.2 — |Peak CCF| Ranking: {label} | black border=significant',
            font=dict(size=12, color=NAVY)),
        xaxis_title='|Peak CCF|',
        xaxis=dict(range=[0, lag_df['|Peak CCF|'].max() + 0.25]),
        template=TEMPLATE,
        height=max(350, len(lag_df) * 35),
        margin=dict(t=60, b=40, l=180, r=220))
    fig2.update_yaxes(autorange='reversed')
    fig2.show()

    # ── Add lagged columns to dataframe ───────────────────────────────────────
    lag_spec = {}
    for _, row in lag_df.iterrows():
        var = row['Variable']
        k   = int(row[f'Peak lag ({unit})'])
        lc  = f'{var}_L{k}'
        df_m[lc] = df_m[var].shift(k)
        lag_spec[var] = {'lag': k, 'col': lc}
    print(f'\nLagged columns added. Dataframe now has {df_m.shape[1]} columns.')
    return lag_df, lag_spec, CONF, N

print('CCF functions defined.')

CCF functions defined.


### 5a — US Quarterly

In [24]:
lag_df_usq, lag_spec_usq, CONF_USQ, N_USQ = ccf_block(
    df_model_usq, TARGET_USQ, FEATURES_STAT_USQ,
    PREFIX_USQ, label='US Quarterly', unit='Q')

SIG_VARS_USQ = lag_df_usq[lag_df_usq['Significant 95%'] == 'Yes']['Variable'].tolist()
print(f'\nSignificant predictors ({len(SIG_VARS_USQ)}): {SIG_VARS_USQ}')
print(f'95% CI threshold: ±{CONF_USQ:.3f}  |  N={N_USQ}')

US Quarterly: N=139, 95% CI=±0.166
Lag windows: Q={'gdp_yoy_growth': 4, 'indprod_yoy': 4, 'consumer_confidence': 4, 'unemployment': 4, 'sp500_log_ret': 4, 'ftse_qoq_growth': 4, 'vix_log_ret': 4, 'reer_diff': 4, 'bond_yield_10y': 4, 'bond_yield_10y_d1': 4, 'house_price_yoy': 6, 'credit_qoq_growth': 6, 'credit_mom_growth': 6, 'cpi': 6} | global max=6


Variable,Max lag window,Peak CCF,|Peak CCF|,Peak lag (Q),Direction,Significant 95%
us_house_price_yoy,6Q,-0.542,0.542,3,Negative,Yes
us_consumer_confidence,4Q,-0.299,0.299,2,Negative,Yes
us_unemployment,4Q,0.283,0.283,1,Positive,Yes
us_credit_qoq_growth,6Q,0.218,0.218,6,Positive,Yes
us_gdp_yoy_growth,4Q,-0.207,0.207,0,Negative,Yes
us_cpi,6Q,0.171,0.171,6,Positive,Yes
us_bond_yield_10y_d1,4Q,-0.155,0.155,2,Negative,No
us_sp500_log_ret,4Q,-0.155,0.155,4,Negative,No
us_reer_diff,4Q,-0.123,0.123,1,Negative,No
us_indprod_yoy,4Q,-0.120,0.120,3,Negative,No



Lagged columns added. Dataframe now has 40 columns.

Significant predictors (6): ['us_house_price_yoy', 'us_consumer_confidence', 'us_unemployment', 'us_credit_qoq_growth', 'us_gdp_yoy_growth', 'us_cpi']
95% CI threshold: ±0.166  |  N=139


### 5b — US Monthly

In [25]:
lag_df_usm, lag_spec_usm, CONF_USM, N_USM = ccf_block(
    df_model_usm, TARGET_USM, FEATURES_STAT_USM,
    PREFIX_USM, label='US Monthly', unit='M')

SIG_VARS_USM = lag_df_usm[lag_df_usm['Significant 95%'] == 'Yes']['Variable'].tolist()
print(f'\nSignificant predictors ({len(SIG_VARS_USM)}): {SIG_VARS_USM}')
print(f'95% CI threshold: ±{CONF_USM:.3f}  |  N={N_USM}')
print('Note: 12/18-month horizons consistent with 4/6-quarter quarterly windows.')

US Monthly: N=415, 95% CI=±0.096
Lag windows: Q={'gdp_yoy_growth': 4, 'indprod_yoy': 4, 'consumer_confidence': 4, 'unemployment': 4, 'sp500_log_ret': 4, 'ftse_qoq_growth': 4, 'vix_log_ret': 4, 'reer_diff': 4, 'bond_yield_10y': 4, 'bond_yield_10y_d1': 4, 'house_price_yoy': 6, 'credit_qoq_growth': 6, 'credit_mom_growth': 6, 'cpi': 6} | global max=18


Variable,Max lag window,Peak CCF,|Peak CCF|,Peak lag (M),Direction,Significant 95%
us_house_price_yoy,18M,-0.541,0.541,7,Negative,Yes
us_unemployment,12M,0.283,0.283,0,Positive,Yes
us_consumer_confidence,12M,-0.275,0.275,3,Negative,Yes
us_credit_mom_growth,18M,0.254,0.254,18,Positive,Yes
us_gdp_yoy_growth,12M,-0.208,0.208,2,Negative,Yes
us_cpi,18M,0.194,0.194,18,Positive,Yes
us_indprod_yoy,12M,-0.116,0.116,5,Negative,Yes
us_sp500_log_ret,12M,-0.110,0.110,12,Negative,Yes
us_bond_yield_10y_d1,12M,-0.107,0.107,4,Negative,Yes
us_reer_diff,12M,-0.093,0.093,0,Negative,No



Lagged columns added. Dataframe now has 40 columns.

Significant predictors (9): ['us_house_price_yoy', 'us_unemployment', 'us_consumer_confidence', 'us_credit_mom_growth', 'us_gdp_yoy_growth', 'us_cpi', 'us_indprod_yoy', 'us_sp500_log_ret', 'us_bond_yield_10y_d1']
95% CI threshold: ±0.096  |  N=415
Note: 12/18-month horizons consistent with 4/6-quarter quarterly windows.


### 5c — UK Quarterly

In [26]:
# gdp_ref = ('uk_gdp_yoy_growth_d1'
#            if 'uk_gdp_yoy_growth_d1' in FEATURES_STAT_UKQ
#            else 'uk_gdp_yoy_growth')
# feat_ukq_no_ref = [f for f in FEATURES_STAT_UKQ if f != gdp_ref]

# lag_df_ukq, lag_spec_ukq, CONF_UKQ, N_UKQ = ccf_block(
#     df_model_ukq, gdp_ref, feat_ukq_no_ref,
#     PREFIX_UKQ, label='UK Quarterly (vs uk_gdp_yoy_growth)', unit='Q')

# lag_spec_ukq[gdp_ref] = {'lag': 0, 'col': gdp_ref}
# SIG_VARS_UKQ = lag_df_ukq[lag_df_ukq['Significant 95%'] == 'Yes']['Variable'].tolist()
# print(f'\nSignificant vs GDP ({len(SIG_VARS_UKQ)}): {SIG_VARS_UKQ}')
# print(f'95% CI threshold: ±{CONF_UKQ:.3f}  |  N={N_UKQ}')
# print('Note: UK significant predictors reflect GDP drivers, not directly default drivers.')

## **6: Multicollinearity Assessment (Correlation + VIF + Clustering)**
___
Assessed at candidate lags from Section 5 — not contemporaneous values — because
multicollinearity must be evaluated as variables will actually enter the model
(Breeden & Crook 2020; Bank & Eder 2021, Table 4, p. 41).

- **Baseline** (VIF < 5): OLS / logistic regression
- **Extended** (VIF < 10): Ridge, Lasso, XGBoost, MLP


In [27]:
def mc_block(df_m, lag_df, lag_spec, sig_vars, all_vars, prefix, label):
    non_sig   = [v for v in all_vars if v not in sig_vars]
    all_lcols = [lag_spec[v]['col'] for v in all_vars if v in lag_spec and lag_spec[v]['col'] in df_m.columns]
    non_lcols = [lag_spec[v]['col'] for v in non_sig  if v in lag_spec and lag_spec[v]['col'] in df_m.columns]
    if not all_lcols:
        print(f'No lagged columns found for {label}')
        return None, [], []
    df_mc = df_m[all_lcols].dropna().copy()
    corr  = df_mc.corr()
    short = [c.replace(prefix, '').replace('_', ' ').title() for c in corr.columns]

    # Correlation heatmap
    fig = go.Figure()
    fig.add_trace(go.Heatmap(z=corr.values, x=short, y=short,
        colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
        text=[[f'{corr.values[i,j]:.2f}' for j in range(len(short))] for i in range(len(short))],
        texttemplate='%{text}', textfont=dict(size=9),
        hovertemplate='%{y} vs %{x}<br>r=%{z:.3f}<extra></extra>',
        colorbar=dict(title='r', thickness=15)))
    for i in range(len(short)):
        for j in range(len(short)):
            if i != j and abs(corr.values[i, j]) > 0.70:
                fig.add_shape(type='rect', x0=j-.5, x1=j+.5, y0=i-.5, y1=i+.5,
                              line=dict(color='gold', width=2.5))
    for idx, col in enumerate(list(corr.columns)):
        if col in non_lcols:
            fig.add_shape(type='rect', x0=idx-.5, x1=idx+.5, y0=-.5, y1=len(corr.columns)-.5,
                line=dict(color=RED, width=2, dash='dash'), fillcolor='rgba(192,57,43,0.05)')
            fig.add_shape(type='rect', x0=-.5, x1=len(corr.columns)-.5, y0=idx-.5, y1=idx+.5,
                line=dict(color=RED, width=2, dash='dash'), fillcolor='rgba(192,57,43,0.05)')
    fig.update_layout(
        title=dict(text=f'Figure 6.1 — Correlation Heatmap: {label} | gold=|r|>0.70 | red dashed=non-sig',
                   font=dict(size=12, color=NAVY)),
        template=TEMPLATE, height=650, width=820, margin=dict(t=110, b=40, l=40, r=40))
    fig.show()

    # VIF table
    X_vif = sm.add_constant(df_mc.copy())
    vif = pd.DataFrame({
        'Variable':   df_mc.columns,
        'VIF':        [variance_inflation_factor(X_vif.values, i+1) for i in range(df_mc.shape[1])],
    })
    vif['VIF']        = vif['VIF'].round(2)
    vif['Group']      = vif['Variable'].apply(lambda v: 'Non-sig' if v in non_lcols else 'Significant')
    vif['Assessment'] = vif['VIF'].apply(
        lambda v: 'OK (VIF<5)' if v < 5 else ('Moderate (5-10)' if v < 10 else 'High >10'))
    vif = vif.sort_values('VIF', ascending=False).reset_index(drop=True)
    display(vif[['Variable', 'Group', 'VIF', 'Assessment']].style
            .map(c_vif, subset=['VIF'])
            .map(c_grp, subset=['Group'])
            .format({'VIF': '{:.2f}'})
            .set_caption(f'Table 6.1 — VIF: {label}')
            .hide(axis='index'))

    # VIF bar chart
    bcols = [GREY if 'Non' in r['Group'] else (GREEN if r['VIF'] < 5 else (AMBER if r['VIF'] < 10 else RED))
             for _, r in vif.iterrows()]
    fig2 = go.Figure(go.Bar(
        x=vif['VIF'],
        y=vif['Variable'].str.replace(prefix, '').str.replace('_', ' ').str.title(),
        orientation='h', marker_color=bcols, marker_line_color=NAVY, marker_line_width=0.8,
        text=vif['VIF'].round(2),
        textposition='outside'))
    fig2.add_vline(x=5,  line_dash='dash', line_color=AMBER, line_width=1.5)
    fig2.add_vline(x=10, line_dash='dash', line_color=RED,   line_width=1.5)
    fig2.update_layout(
        title=dict(text=f'Figure 6.2 — VIF: {label}', font=dict(size=13, color=NAVY)),
        xaxis_title='VIF', xaxis=dict(range=[0, vif['VIF'].max() + 3]),
        template=TEMPLATE, height=500, margin=dict(t=80, b=40, l=220, r=120))
    fig2.update_yaxes(autorange='reversed')
    fig2.show()

    # Hierarchical clustering dendrogram
    dist = np.clip(1 - np.abs(corr.values), 0, None)
    np.fill_diagonal(dist, 0)
    Z   = linkage(squareform(dist), method='ward')
    CUT = 0.45
    fig3, ax = plt.subplots(figsize=(14, 5))
    scipy_dendrogram(Z, labels=short, ax=ax, leaf_rotation=45, leaf_font_size=9, color_threshold=CUT)
    ax.axhline(CUT, color=RED, lw=1.5, ls='--', label=f'Cut-off ({CUT})')
    ax.set_title(f'Figure 6.3 — Dendrogram: {label}', fontsize=10, color=NAVY)
    ax.set_ylabel('Distance (1 − |r|)')
    ax.set_facecolor('#F8F9FA')
    fig3.patch.set_facecolor('white')
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    FB = vif[(vif['VIF'] < 5)  & (vif['Group'] == 'Significant')]['Variable'].tolist()
    FE = vif[(vif['VIF'] < 10) & (vif['Group'] == 'Significant')]['Variable'].tolist()
    print(f'Baseline ({len(FB)}, VIF<5):  {FB}')
    print(f'Extended ({len(FE)}, VIF<10): {FE}')
    return vif, FB, FE

print('Multicollinearity functions defined.')


Multicollinearity functions defined.


### 6a — US Quarterly

In [28]:
vif_usq, FB_USQ, FE_USQ = mc_block(
    df_model_usq, lag_df_usq, lag_spec_usq,
    SIG_VARS_USQ, lag_df_usq['Variable'].tolist(), PREFIX_USQ, 'US Quarterly')

Variable,Group,VIF,Assessment
us_unemployment_L1,Significant,2.46,OK (VIF<5)
us_consumer_confidence_L2,Significant,2.36,OK (VIF<5)
us_house_price_yoy_L3,Significant,1.90,OK (VIF<5)
us_indprod_yoy_L3,Non-sig,1.84,OK (VIF<5)
us_credit_qoq_growth_L6,Significant,1.63,OK (VIF<5)
us_gdp_yoy_growth_L0,Significant,1.46,OK (VIF<5)
us_oil_yoy_L2,Non-sig,1.41,OK (VIF<5)
us_cpi_L6,Significant,1.37,OK (VIF<5)
us_bond_yield_10y_d1_L2,Non-sig,1.26,OK (VIF<5)
us_sp500_log_ret_L4,Non-sig,1.24,OK (VIF<5)


Baseline (6, VIF<5):  ['us_unemployment_L1', 'us_consumer_confidence_L2', 'us_house_price_yoy_L3', 'us_credit_qoq_growth_L6', 'us_gdp_yoy_growth_L0', 'us_cpi_L6']
Extended (6, VIF<10): ['us_unemployment_L1', 'us_consumer_confidence_L2', 'us_house_price_yoy_L3', 'us_credit_qoq_growth_L6', 'us_gdp_yoy_growth_L0', 'us_cpi_L6']


### 6b — US Monthly

In [29]:
vif_usm, FB_USM, FE_USM = mc_block(
    df_model_usm, lag_df_usm, lag_spec_usm,
    SIG_VARS_USM, lag_df_usm['Variable'].tolist(), PREFIX_USM, 'US Monthly')

Variable,Group,VIF,Assessment
us_gdp_yoy_growth_L2,Significant,3.61,OK (VIF<5)
us_indprod_yoy_L5,Significant,2.65,OK (VIF<5)
us_unemployment_L0,Significant,2.54,OK (VIF<5)
us_consumer_confidence_L3,Significant,2.30,OK (VIF<5)
us_house_price_yoy_L7,Significant,1.84,OK (VIF<5)
us_credit_mom_growth_L18,Significant,1.60,OK (VIF<5)
us_oil_yoy_L4,Non-sig,1.43,OK (VIF<5)
us_cpi_L18,Significant,1.28,OK (VIF<5)
us_bond_yield_10y_d1_L4,Significant,1.17,OK (VIF<5)
us_sp500_log_ret_L12,Significant,1.14,OK (VIF<5)


Baseline (9, VIF<5):  ['us_gdp_yoy_growth_L2', 'us_indprod_yoy_L5', 'us_unemployment_L0', 'us_consumer_confidence_L3', 'us_house_price_yoy_L7', 'us_credit_mom_growth_L18', 'us_cpi_L18', 'us_bond_yield_10y_d1_L4', 'us_sp500_log_ret_L12']
Extended (9, VIF<10): ['us_gdp_yoy_growth_L2', 'us_indprod_yoy_L5', 'us_unemployment_L0', 'us_consumer_confidence_L3', 'us_house_price_yoy_L7', 'us_credit_mom_growth_L18', 'us_cpi_L18', 'us_bond_yield_10y_d1_L4', 'us_sp500_log_ret_L12']


### 6c — UK Quarterly

In [30]:
# vif_ukq, FB_UKQ, FE_UKQ = mc_block(
#     df_model_ukq, lag_df_ukq, lag_spec_ukq,
#     SIG_VARS_UKQ, lag_df_ukq['Variable'].tolist(), PREFIX_UKQ, 'UK Quarterly')

## **7: ACF/PACF Analysis**
___
Dual purpose: (1) ARIMA order selection for Stage 1 macro forecasting (BDO Research
Question 6); (2) target persistence documentation for Stage 2 model specification.

ACF/PACF is run on the **unlagged, stationary** versions of each variable — these are
what get their own ARIMA models in Stage 1.

**Reading the plots:**
- **ACF (blue):** Autocorrelation at each lag — how correlated is the series with its
  own past values
- **PACF (amber):** Partial autocorrelation — correlation at each lag after removing
  the effect of shorter lags
- **Red dotted lines:** 95% confidence interval — bars beyond these are significant

**ARIMA order selection rules:**
- PACF cuts off, ACF decays slowly → **AR process** → use AR(p)
- ACF cuts off, PACF decays slowly → **MA process** → use MA(q)
- Both decay gradually → **ARMA process** → use ARMA(p,q)
- Neither significant → **White noise** → AR(1) as baseline

**Monthly dataset uses 24 lags** — check lag 12 for seasonal spikes indicating SARIMA.

**References:**
Djeundje, V.B. & Crook, J. (2019). Dynamic survival models with varying coefficients
for credit risks. *European Journal of Operational Research*, 275(1), 319–333.

Bank, M. & Eder, B. (2021). A Review on the Probability of Default for IFRS 9.
SSRN Working Paper 3981339.

In [31]:
def acf_pacf_target(df_m, target, nlags, label):
    s    = df_m[target].dropna()
    n    = len(s)
    conf = 1.96 / np.sqrt(n)
    av   = acf(s,  nlags=nlags, fft=True,   alpha=None)
    pv   = pacf(s, nlags=nlags, method='ywm', alpha=None)
    lags_arr = np.arange(0, nlags + 1)
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'ACF — {target}', f'PACF — {target}'],
        horizontal_spacing=0.12)
    for ci, (vals, nm) in enumerate([(av, 'ACF'), (pv, 'PACF')], 1):
        fig.add_trace(go.Bar(
            x=lags_arr, y=vals,
            marker_color=[NAVY if i == 0 else (BLUE if abs(v) > conf else LBLUE)
                          for i, v in enumerate(vals)],
            showlegend=False,
            hovertemplate=f'Lag %{{x}}<br>{nm}=%{{y:.3f}}<extra></extra>'),
            row=1, col=ci)
        fig.add_hline(y= conf, line_dash='dot', line_color=RED,
                      line_width=1.5, row=1, col=ci)
        fig.add_hline(y=-conf, line_dash='dot', line_color=RED,
                      line_width=1.5, row=1, col=ci)
        fig.add_hline(y=0, line_color='black', line_width=0.5, row=1, col=ci)
    fig.update_xaxes(title_text='Lag', tickvals=list(lags_arr),
                     tickfont=dict(size=7))
    fig.update_yaxes(tickfont=dict(size=7))
    fig.update_annotations(font_size=9)
    fig.update_layout(
        title=dict(
            text=f'Figure 7.1 — ACF & PACF: {target} ({label}) | red dotted=95% CI',
            font=dict(size=12, color=NAVY)),
        template=TEMPLATE,
        height=280,
        margin=dict(t=80, b=30, l=40, r=30))
    fig.show()


def acf_pacf_features(df_m, sig_vars, prefix, nlags, label):
    if not sig_vars: return
    lags_arr = np.arange(0, nlags + 1)
    nrows    = len(sig_vars)
    fig = make_subplots(
        rows=nrows, cols=2,
        subplot_titles=[item for v in sig_vars
            for item in [
                v.replace(prefix, '').replace('_', ' ').title() + ' — ACF',
                v.replace(prefix, '').replace('_', ' ').title() + ' — PACF']],
        vertical_spacing=min(0.08, 0.9 / nrows),
        horizontal_spacing=0.10)
    for i, var in enumerate(sig_vars):
        row  = i + 1
        s    = df_m[var].dropna()
        c    = 1.96 / np.sqrt(len(s))
        av   = acf(s,  nlags=nlags, fft=True,   alpha=None)
        pv   = pacf(s, nlags=nlags, method='ywm', alpha=None)
        for ci, (vals, cs, cn) in enumerate(
                [(av, BLUE, LBLUE), (pv, AMBER, '#FAD7A0')], 1):
            fig.add_trace(go.Bar(
                x=lags_arr, y=vals,
                marker_color=[NAVY if j == 0 else (cs if abs(v) > c else cn)
                              for j, v in enumerate(vals)],
                showlegend=False), row=row, col=ci)
            fig.add_hline(y= c, line_dash='dot', line_color=RED,
                          line_width=1, row=row, col=ci)
            fig.add_hline(y=-c, line_dash='dot', line_color=RED,
                          line_width=1, row=row, col=ci)
            fig.add_hline(y=0, line_color='black', line_width=0.5,
                          row=row, col=ci)
    fig.update_xaxes(tickvals=list(lags_arr), tickfont=dict(size=7))
    fig.update_yaxes(tickfont=dict(size=7))
    fig.update_annotations(font_size=8)
    fig.update_layout(
        title=dict(
            text=(f'Figure 7.2 — ACF (blue) & PACF (amber): '
                  f'Significant Predictors ({label})'),
            font=dict(size=12, color=NAVY)),
        template=TEMPLATE,
        height=nrows * 160,
        margin=dict(t=80, b=30, l=40, r=30))
    fig.show()


def arima_table(df_m, target, sig_vars, prefix, nlags, label):
    def suggest(series, name):
        s    = series.dropna()
        n    = len(s)
        conf = 1.96 / np.sqrt(n)
        av   = acf(s,  nlags=nlags, fft=True,   alpha=None)
        pv   = pacf(s, nlags=nlags, method='ywm', alpha=None)
        as_  = [i for i in range(1, nlags+1) if abs(av[i]) > conf]
        ps_  = [i for i in range(1, nlags+1) if abs(pv[i]) > conf]
        pm, px = (ps_[0], min(ps_[-1], 5)) if ps_ else (0, 0)
        qm, qx = (as_[0], min(as_[-1], 5)) if as_ else (0, 0)
        aslow, pslow = len(as_) > 4, len(ps_) > 4
        if aslow and not pslow:
            pat = 'AR process — PACF cuts off'
            sug = f'AR({pm}) — test to AR({px})' if pm != px else f'AR({pm})'
        elif pslow and not aslow:
            pat = 'MA process — ACF cuts off'
            sug = f'MA({qm}) — test to MA({qx})' if qm != qx else f'MA({qm})'
        elif aslow and pslow:
            pat = 'ARMA — both decay gradually'
            sug = f'ARMA({pm},{qm}) — test to ARMA({px},{qx})'
        else:
            pat = 'White noise / short memory'
            sug = 'AR(1) or white noise'
        return {
            'Variable': name,
            'N':        len(s),
            'Sig ACF':  len(as_),
            'Sig PACF': len(ps_),
            'p range':  f'{pm}–{px}' if pm != px else str(pm),
            'q range':  f'{qm}–{qx}' if qm != qx else str(qm),
            'Suggested': sug,
            'Pattern':   pat,
        }
    rows = []
    if target and target in df_m.columns:
        rows.append(suggest(df_m[target], target))
    for var in sig_vars:
        rows.append(suggest(
            df_m[var],
            var.replace(prefix, '').replace('_', ' ').title()))
    ard = pd.DataFrame(rows)
    display(ard.style
            .map(c_pattern,  subset=['Pattern'])
            .map(c_siglags,  subset=['Sig ACF', 'Sig PACF'])
            .format({'N': '{:.0f}'})
            .set_caption(
                f'Table 7.1 — Suggested ARIMA Orders: {label} | direct input to Stage 1')
            .hide(axis='index'))
    return ard

print('ACF/PACF functions defined.')

ACF/PACF functions defined.


### 7a — US Quarterly

In [32]:
acf_pacf_target(df_model_usq, TARGET_USQ, 12, 'US Quarterly')
acf_pacf_features(df_model_usq, SIG_VARS_USQ, PREFIX_USQ, 12, 'US Quarterly')
arima_df_usq = arima_table(
    df_model_usq, TARGET_USQ, SIG_VARS_USQ, PREFIX_USQ, 12, 'US Quarterly')

Variable,N,Sig ACF,Sig PACF,p range,q range,Suggested,Pattern
us_delinquency_rate,140,12,2,1–2,1–5,AR(1) — test to AR(2),AR process — PACF cuts off
House Price Yoy,140,12,3,1–5,1–5,AR(1) — test to AR(5),AR process — PACF cuts off
Consumer Confidence,144,8,4,1–4,1–5,AR(1) — test to AR(4),AR process — PACF cuts off
Unemployment,144,10,1,1,1–5,AR(1),AR process — PACF cuts off
Credit Qoq Growth,143,10,4,1–5,1–5,AR(1) — test to AR(5),AR process — PACF cuts off
Gdp Yoy Growth,144,3,4,1–5,1–3,AR(1) or white noise,White noise / short memory
Cpi,143,5,6,1–5,1–5,"ARMA(1,1) — test to ARMA(5,5)",ARMA — both decay gradually


### 7b — US Monthly

In [33]:
acf_pacf_target(df_model_usm, TARGET_USM, 24, 'US Monthly')
acf_pacf_features(df_model_usm, SIG_VARS_USM, PREFIX_USM, 24, 'US Monthly')
arima_df_usm = arima_table(
    df_model_usm, TARGET_USM, SIG_VARS_USM, PREFIX_USM, 24, 'US Monthly')
print('Check lag 12 in ACF — seasonal spike indicates SARIMA needed.')

Variable,N,Sig ACF,Sig PACF,p range,q range,Suggested,Pattern
us_delinquency_rate,418,24,3,1–3,1–5,AR(1) — test to AR(3),AR process — PACF cuts off
House Price Yoy,422,24,3,1–3,1–5,AR(1) — test to AR(3),AR process — PACF cuts off
Unemployment,432,24,6,1–5,1–5,"ARMA(1,1) — test to ARMA(5,5)",ARMA — both decay gradually
Consumer Confidence,434,21,3,1–5,1–5,AR(1) — test to AR(5),AR process — PACF cuts off
Credit Mom Growth,429,24,15,1–5,1–5,"ARMA(1,1) — test to ARMA(5,5)",ARMA — both decay gradually
Gdp Yoy Growth,432,10,12,1–5,1–5,"ARMA(1,1) — test to ARMA(5,5)",ARMA — both decay gradually
Cpi,429,19,10,1–5,1–5,"ARMA(1,1) — test to ARMA(5,5)",ARMA — both decay gradually
Indprod Yoy,422,11,9,1–5,1–5,"ARMA(1,1) — test to ARMA(5,5)",ARMA — both decay gradually
Sp500 Log Ret,434,0,0,0,0,AR(1) or white noise,White noise / short memory
Bond Yield 10Y D1,433,1,3,1–5,1,AR(1) or white noise,White noise / short memory


Check lag 12 in ACF — seasonal spike indicates SARIMA needed.


### 7c — UK Quarterly

In [34]:
# acf_pacf_features(df_model_ukq, FEATURES_STAT_UKQ, PREFIX_UKQ, 12, 'UK Quarterly')
# arima_df_ukq = arima_table(
#     df_model_ukq, None, FEATURES_STAT_UKQ, PREFIX_UKQ, 12, 'UK Quarterly')

## **8: Cross-Dataset Comparison & Final Summary**
___
This section documents every column present in each final dataframe and summarises all analytical decisions made throughout the notebook.

**Dataframe contents after full run:**
- **Unlagged pipeline vars** — stationary features entering the analytical pipeline
- **Lagged versions (_Lk)** — added by CCF at the data-driven optimal lag per variable;
  these are the intended inputs for Stage 2 PD modelling
- **Excluded vars** — retained in the dataframe, not in the pipeline
- **Other** — target variable, `_d1` first-differenced columns

**Stage routing:**
- Unlagged versions → Stage 1 ARIMA/ML macro forecasting
- Lagged versions → Stage 2 PD regression

**UK dataset limitation:** No aggregate default series available at quarterly frequency back to 1990. UK results support Stage 1 macro forecasting only. Stage 2 PD modelling requires account-level data from UK credit bureaus (Experian, Equifax, TransUnion) as per Bank & Eder (2021, Section 2, p. 3).

In [35]:
# Table 8.1 — Cross-dataset coverage summary
rows = [
    {'Dataset':          'US Quarterly',
     'Freq':             'Q',
     'Target':           'us_delinquency_rate',
     'Obs':              len(df_model_usq),
     'From':             str(df_model_usq.index.min().date()),
     'To':               str(df_model_usq.index.max().date()),
     'Total cols in df': df_model_usq.shape[1],
     'In pipeline':      len(FEATURES_STAT_USQ),
     'Sig. predictors':  len(SIG_VARS_USQ),
     'Stage 2 ready':    'Yes'},
    {'Dataset':          'US Monthly',
     'Freq':             'M',
     'Target':           'us_delinquency_rate (spline)',
     'Obs':              len(df_model_usm),
     'From':             str(df_model_usm.index.min().date()),
     'To':               str(df_model_usm.index.max().date()),
     'Total cols in df': df_model_usm.shape[1],
     'In pipeline':      len(FEATURES_STAT_USM),
     'Sig. predictors':  len(SIG_VARS_USM),
     'Stage 2 ready':    'Yes'}
    # {'Dataset':          'UK Quarterly',
    #  'Freq':             'Q',
    #  'Target':           'None — Stage 1 only',
    #  'Obs':              len(df_model_ukq),
    #  'From':             str(df_model_ukq.index.min().date()),
    #  'To':               str(df_model_ukq.index.max().date()),
    #  'Total cols in df': df_model_ukq.shape[1],
    #  'In pipeline':      len(FEATURES_STAT_UKQ),
    #  'Sig. predictors':  len(SIG_VARS_UKQ),
    #  'Stage 2 ready':    'No — no target variable'},
]

def c_s2(v):
    if v == 'Yes': return 'background-color:#D5F5E3; color:#1a1a1a; font-weight:bold'
    return 'background-color:#FADBD8; color:#1a1a1a'

display(pd.DataFrame(rows).style
        .map(c_s2, subset=['Stage 2 ready'])
        .set_caption('Table 8.1 — Cross-Dataset Coverage Summary')
        .hide(axis='index'))

Dataset,Freq,Target,Obs,From,To,Total cols in df,In pipeline,Sig. predictors,Stage 2 ready
US Quarterly,Q,us_delinquency_rate,144,1990-03-31,2025-12-31,40,12,6,Yes
US Monthly,M,us_delinquency_rate (spline),434,1990-01-31,2026-02-28,40,12,9,Yes


In [36]:
# Figure 8.1 — Peak CCF comparison: US Quarterly vs US Monthly
# UK excluded — CCF was vs GDP not vs default target
usq_map = dict(zip(
    lag_df_usq['Variable'].str.replace(PREFIX_USQ, ''),
    lag_df_usq['|Peak CCF|']))
usm_map = dict(zip(
    lag_df_usm['Variable'].str.replace(PREFIX_USM, ''),
    lag_df_usm['|Peak CCF|']))
all_short = sorted(set(list(usq_map) + list(usm_map)))

fig = go.Figure()
fig.add_trace(go.Bar(
    name='US Quarterly', y=all_short,
    x=[usq_map.get(v, 0) for v in all_short],
    orientation='h', marker_color=BLUE, opacity=0.8))
fig.add_trace(go.Bar(
    name='US Monthly', y=all_short,
    x=[usm_map.get(v, 0) for v in all_short],
    orientation='h', marker_color=AMBER, opacity=0.8))
fig.add_vline(x=CONF_USQ, line_dash='dash', line_color=BLUE,
              line_width=1, annotation_text=f'CI USQ={CONF_USQ:.2f}',
              annotation_font_color=BLUE, annotation_font_size=8)
fig.add_vline(x=CONF_USM, line_dash='dash', line_color=AMBER,
              line_width=1, annotation_text=f'CI USM={CONF_USM:.2f}',
              annotation_font_color=AMBER, annotation_font_size=8)
fig.update_layout(
    barmode='group',
    title=dict(
        text='Figure 8.1 — |Peak CCF| vs Target: US Quarterly vs Monthly<br>'
             '<sup>UK excluded — CCF was run vs GDP growth, not vs a default target</sup>',
        font=dict(size=12, color=NAVY)),
    xaxis_title='|Peak CCF|',
    xaxis=dict(range=[0, max(max(usq_map.values()), max(usm_map.values())) + 0.15]),
    template=TEMPLATE,
    height=max(350, len(all_short) * 30),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=100, b=40, l=180, r=40))
fig.update_yaxes(autorange='reversed')
fig.show()

In [37]:
def variable_inventory(df_m, features, lag_df, lag_spec,
                        excl_dict, prefix, label, has_target=True):
    unit         = 'M' if 'Monthly' in label else 'Q'
    lag_col_name = f'Peak lag ({unit})'
    rows         = []

    # Retrieve threshold from N
    N      = len(df_m[[f for f in features if f in df_m.columns]].dropna())
    thresh = round(1.96 / np.sqrt(N), 3)

    lag_lookup = {}
    if lag_df is not None:
        for _, row in lag_df.iterrows():
            var = row['Variable']
            lag_lookup[var] = {
                'lag': int(row[lag_col_name]),
                'ccf': round(row['Peak CCF'], 3),
                'sig': row['Significant 95%'],
                'col': lag_spec[var]['col'] if var in lag_spec else '—',
            }

    # ── Pipeline variables ────────────────────────────────────────────────────
    for var in features:
        info    = lag_lookup.get(var, {})
        lag_k   = info.get('lag', '—')
        lag_col = info.get('col', '—')
        ccf_val = info.get('ccf', '—')
        sig     = info.get('sig', '—')
        in_df   = lag_col if lag_col != '—' and lag_col in df_m.columns else '—'

        if has_target:
            pd_pred = in_df  # always show lagged col regardless of significance
        else:
            pd_pred = 'No target'

        rows.append({
            'Variable (unlagged)': var,
            'Opt. Lag':            lag_k,
            'Lagged Col':          in_df,
            'Peak CCF':            ccf_val,
            'Significant':         sig,
            '_sig_flag':           sig,
            'Macro Forecast':      'Ready to use',
            'PD Prediction':       pd_pred,
        })

    # ── Excluded variables ────────────────────────────────────────────────────
    for var, reason in excl_dict.items():
        if var not in df_m.columns: continue
        cat = reason.split(' — ')[0]
        rows.append({
            'Variable (unlagged)': var,
            'Opt. Lag':            '—',
            'Lagged Col':          '—',
            'Peak CCF':            '—',
            'Significant':         '—',
            '_sig_flag':           '—',
            'Macro Forecast':      f'✗ {cat}',
            'PD Prediction':       '—',
        })

    inv = pd.DataFrame(rows).reset_index(drop=True)

    # ── Stylers ───────────────────────────────────────────────────────────────
    def c_ccf(v):
        try:
            val = abs(float(v))
            if val >= thresh:        return 'background-color:#D5F5E3; color:#1a1a1a; font-weight:bold'
            if val >= thresh * 0.75: return 'background-color:#FEF9E7; color:#1a1a1a'
            return                          'background-color:#FADBD8; color:#1a1a1a'
        except:
            return 'color:#7F8C8D'

    def c_sig(v):
        if v == 'Yes': return 'background-color:#D5F5E3; color:#1a1a1a; font-weight:bold'
        if v == 'No':  return 'background-color:#FADBD8; color:#1a1a1a'
        return 'color:#7F8C8D'

    def c_macro(v):
        if v == 'Ready to use': return 'background-color:#EBF5FB; color:#1a1a1a'
        return 'background-color:#F2F3F4; color:#7F8C8D'

    def c_pd_row(row):
        styles = ['' for _ in row.index]
        pd_idx = row.index.get_loc('PD Prediction')
        v      = row['PD Prediction']
        sig    = inv.loc[row.name, '_sig_flag']
        if v == 'No target':
            styles[pd_idx] = 'background-color:#FEF9E7; color:#1a1a1a'
        elif v == '—':
            styles[pd_idx] = 'color:#7F8C8D'
        elif sig == 'Yes':
            styles[pd_idx] = 'background-color:#D5F5E3; color:#1a1a1a; font-weight:bold'
        else:
            styles[pd_idx] = 'background-color:#FEF9E7; color:#1a1a1a'
        return styles

    def fmt_ccf(v):
        try:    return f'{float(v):+.3f}'
        except: return v

    display(inv.drop(columns=['_sig_flag']).style
            .map(c_ccf,   subset=['Peak CCF'])
            .map(c_sig,   subset=['Significant'])
            .map(c_macro, subset=['Macro Forecast'])
            .apply(c_pd_row, axis=1)
            .format({'Peak CCF': fmt_ccf})
            .set_caption(
                f'Table 8.2 — Variable Inventory: {label} | '
                f'95% CI threshold = ±{thresh} (N≈{N}) | '
                f'Peak CCF is a correlation coefficient [-1, +1] — '
                f'green = significant | yellow = below threshold')
            .hide(axis='index'))

    # ── Column count summary ──────────────────────────────────────────────────
    lagged_cols  = [c for c in df_m.columns
                    if '_L' in c and c.split('_L')[-1].isdigit()]
    excl_present = [v for v in excl_dict if v in df_m.columns]
    print(f'\n{label} — {df_m.shape[1]} columns in df:')
    print(f'  Pipeline (unlagged) : {len(features)}')
    print(f'  Lagged (_Lk)        : {len(lagged_cols)}')
    print(f'  Excluded (retained) : {len(excl_present)}')
    other = df_m.shape[1] - len(features) - len(lagged_cols) - len(excl_present)
    print(f'  Other               : {other}')


# ── US Quarterly ──────────────────────────────────────────────────────────────
variable_inventory(df_model_usq, FEATURES_STAT_USQ, lag_df_usq, lag_spec_usq,
                   EXCL_USQ, PREFIX_USQ, 'US Quarterly', has_target=True)

# ── US Monthly ────────────────────────────────────────────────────────────────
variable_inventory(df_model_usm, FEATURES_STAT_USM, lag_df_usm, lag_spec_usm,
                   EXCL_USM, PREFIX_USM, 'US Monthly', has_target=True)

# ── UK Quarterly — Stage 1 only, no default target ───────────────────────────
# variable_inventory(df_model_ukq, FEATURES_STAT_UKQ, lag_df_ukq, lag_spec_ukq,
#                    EXCL_UKQ, PREFIX_UKQ, 'UK Quarterly', has_target=False)

Variable (unlagged),Opt. Lag,Lagged Col,Peak CCF,Significant,Macro Forecast,PD Prediction
us_gdp_yoy_growth,0,us_gdp_yoy_growth_L0,-0.207,Yes,Ready to use,us_gdp_yoy_growth_L0
us_unemployment,1,us_unemployment_L1,+0.283,Yes,Ready to use,us_unemployment_L1
us_cpi,6,us_cpi_L6,+0.171,Yes,Ready to use,us_cpi_L6
us_consumer_confidence,2,us_consumer_confidence_L2,-0.299,Yes,Ready to use,us_consumer_confidence_L2
us_bond_yield_10y_d1,2,us_bond_yield_10y_d1_L2,-0.155,No,Ready to use,us_bond_yield_10y_d1_L2
us_credit_qoq_growth,6,us_credit_qoq_growth_L6,+0.218,Yes,Ready to use,us_credit_qoq_growth_L6
us_sp500_log_ret,4,us_sp500_log_ret_L4,-0.155,No,Ready to use,us_sp500_log_ret_L4
us_vix_log_ret,0,us_vix_log_ret_L0,-0.021,No,Ready to use,us_vix_log_ret_L0
us_house_price_yoy,3,us_house_price_yoy_L3,-0.542,Yes,Ready to use,us_house_price_yoy_L3
us_indprod_yoy,3,us_indprod_yoy_L3,-0.120,No,Ready to use,us_indprod_yoy_L3



US Quarterly — 40 columns in df:
  Pipeline (unlagged) : 12
  Lagged (_Lk)        : 12
  Excluded (retained) : 14
  Other               : 2


Variable (unlagged),Opt. Lag,Lagged Col,Peak CCF,Significant,Macro Forecast,PD Prediction
us_gdp_yoy_growth,2,us_gdp_yoy_growth_L2,-0.208,Yes,Ready to use,us_gdp_yoy_growth_L2
us_unemployment,0,us_unemployment_L0,+0.283,Yes,Ready to use,us_unemployment_L0
us_cpi,18,us_cpi_L18,+0.194,Yes,Ready to use,us_cpi_L18
us_consumer_confidence,3,us_consumer_confidence_L3,-0.275,Yes,Ready to use,us_consumer_confidence_L3
us_bond_yield_10y_d1,4,us_bond_yield_10y_d1_L4,-0.107,Yes,Ready to use,us_bond_yield_10y_d1_L4
us_credit_mom_growth,18,us_credit_mom_growth_L18,+0.254,Yes,Ready to use,us_credit_mom_growth_L18
us_sp500_log_ret,12,us_sp500_log_ret_L12,-0.110,Yes,Ready to use,us_sp500_log_ret_L12
us_vix_log_ret,11,us_vix_log_ret_L11,+0.016,No,Ready to use,us_vix_log_ret_L11
us_house_price_yoy,7,us_house_price_yoy_L7,-0.541,Yes,Ready to use,us_house_price_yoy_L7
us_indprod_yoy,5,us_indprod_yoy_L5,-0.116,Yes,Ready to use,us_indprod_yoy_L5



US Monthly — 40 columns in df:
  Pipeline (unlagged) : 12
  Lagged (_Lk)        : 12
  Excluded (retained) : 14
  Other               : 2


In [38]:
# Save complete analytical datasets — all columns, all rows
df_model_usq.to_csv(BASE / 'EDA_analytical_US_Q.csv')
df_model_usm.to_csv(BASE / 'EDA_analytical_US_M.csv')
df_model_ukq.to_csv(BASE / 'EDA_analytical_UK_Q.csv')
print('Saved:')
print(f'  EDA_analytical_US_Q.csv  {df_model_usq.shape}')
print(f'  EDA_analytical_US_M.csv  {df_model_usm.shape}')
print(f'  EDA_analytical_UK_Q.csv  {df_model_ukq.shape}')

Saved:
  EDA_analytical_US_Q.csv  (144, 40)
  EDA_analytical_US_M.csv  (434, 40)
  EDA_analytical_UK_Q.csv  (144, 21)
